In [61]:
# Célula 0 — Imports e configuração
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.cloud import bigquery

# === Paleta de Cores (semântica explícita: fluxo vs status) ===
COLORS = {
    # Cores de fluxo (identidade, não performance)
    "fluxo_pa":   "#6B46C1",   # Roxo — Produto Acabado
    "fluxo_tri":  "#EAB308",   # Amarelo — Triangulação

    # Cores de status (performance vs target)
    "status_ok":       "#16A34A",  # Verde — desvio ≤10d ou dentro do prazo
    "status_atencao":  "#F97316",  # Laranja — desvio 10–30d
    "status_critico":  "#DC2626",  # Vermelho — desvio >30d

    # Neutros
    "neutro_escuro":   "#374151",  # Cinza-escuro — texto, eixos
    "neutro_claro":    "#E5E7EB",  # Cinza-claro — células sem amostra
    "target_line":     "#9CA3AF",  # Cinza tracejado — linha de SLA
}
TEMPLATE = "plotly_white"
SLA_ETAPAS = {
    "Reserva → Recebimento Tecido": 65,
    "Produção → Auditoria": 45,
    "Auditoria → Entrega CD": 14,
}
TARGET_LEAD_TIME = sum(SLA_ETAPAS.values())  # 124 dias — soma dos SLAs padrão

# === Configuração de Análise (parametrizável) ===
CONFIG = {
    # Janela temporal padrão (em meses corridos a partir de hoje)
    "janela_meses": 12,

    # Janela curta para comparação (tendência recente)
    "janela_meses_curta": 3,

    # Mínimo de OPs para incluir um agregado
    "min_ops_recomendacao": 5,
    "min_ops_grafico":      3,

    # Percentil usado para calcular o prazo recomendado por fornecedor×fluxo
    # 0.50 = mediana (agressivo) | 0.75 = padrão | 0.90 = conservador
    "percentil_recomendacao": 0.75,

    # Thresholds de status (em dias)
    "threshold_desvio_atencao": 10,
    "threshold_desvio_critico": 30,

    # Buckets de volume (faixas absolutas, conforme metodologia do relatório)
    "buckets_volume": [0, 200, 500, 1000, 2000, 5000, float("inf")],
    "labels_volume": ["<200", "200-499", "500-999", "1000-1999", "2000-4999", "5000+"],

    # Quantos fornecedores mostrar nos gráficos com ranking
    "top_n_fornecedores_grafico": 15,
    "top_n_fornecedores_heatmap": 20,
}

PROJECT_ID = "insider-data-lake"  # ajuste se necessário

try:
    client = bigquery.Client(project=PROJECT_ID)
    print(f"✅ Conectado ao BigQuery — projeto: {client.project}")
except Exception as e:
    print(f"❌ Falha na conexão com BigQuery: {e}")
    print("Execute 'gcloud auth application-default login' no terminal e reinicie o kernel.")
    raise


/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


✅ Conectado ao BigQuery — projeto: insider-data-lake


In [62]:
# Célula 1 — Base de capacidade (fornecedores, produtos, lead time teórico, mark-up)
SQL_CAPACITY = """
WITH
full_price AS (
    SELECT *
    FROM `insider-data-lake.integrated.muninn_products`
),
fabric_costs AS(
    SELECT 
    mfs.id AS fabric_sku_id,
    mfs.fabric_id,
    mfs.knitting_factory_id,
    mfs.sku AS fabric_sku,
    mfs.invoice_fabric_name AS factory_fabric_name,
    mfs.unit_price,
    mfs.minimum_volume_per_order,
    mfs.multiple_volume_per_order,
    mf.name AS fabric_name,
    mf.article_id,
    ma.name AS article_name,
    ma.unit AS article_unit,
    mkf.supplier_id,
    ms.alias AS knitting_factory_name,
    FROM `insider-data-lake.integrated.muninn_fabric_skus` AS mfs
    LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mfs.fabric_id
    LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id
    LEFT JOIN `insider-data-lake.integrated.muninn_knitting_factories` AS mkf ON mkf.id = mfs.knitting_factory_id
    LEFT JOIN `insider-data-lake.integrated.muninn_suppliers` AS ms ON ms.id = mkf.supplier_id
    WHERE mfs.status IN ('available')
    ),
fabric_min_max_cost AS (
    SELECT
    fc.fabric_id,
    fc.fabric_name,
    MIN(fc.unit_price) AS min_fabric_cost,
    MAX(fc.unit_price) AS max_fabric_cost,
    COUNT(DISTINCT fc.knitting_factory_id) AS number_knitting_factories,
    ARRAY_AGG(DISTINCT fc.knitting_factory_name) AS knitting_factories_names,
    FROM fabric_costs AS fc
    GROUP BY fc.fabric_id,
    fc.fabric_name
),
article_sku AS (
SELECT
    mpsf.product_sku_id,
    mps.sku,
    mps.sku_name,
    mps.product_id,
    s.sku_state,
    s.gender,
    s.color,
    s.size,
    s.product_name,
    mpsf.fabric_id,
    mf.name AS fabric_name,
    mpsf.consumption,
    fc.min_fabric_cost AS min_fabric_unitary_cost,
    fc.max_fabric_cost AS max_fabric_unitary_cost,
    fc.min_fabric_cost * mpsf.consumption AS min_fabric_cost,
    fc.max_fabric_cost * mpsf.consumption AS max_fabric_cost,
    ma.unit AS article_unit,
    ma.name AS article_name,
    mf.article_id,
    fc.number_knitting_factories,
    fc.knitting_factories_names
FROM `insider-data-lake.integrated.muninn_product_skus_fabrics` AS mpsf
LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mpsf.fabric_id
LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id
LEFT JOIN `insider-data-lake.integrated.muninn_product_skus` AS mps ON mps.product_sku_id = mpsf.product_sku_id
LEFT JOIN `insider-data-lake.integrated.skus` AS s ON mps.sku = s.sku
LEFT JOIN fabric_min_max_cost AS fc ON fc.fabric_id = mpsf.fabric_id
),
sku_fabric_costs AS (
    SELECT 
        a_sku.sku, a_sku.sku_state, a_sku.product_id,
        SUM(a_sku.min_fabric_cost) AS min_fabric_cost,
        SUM(a_sku.max_fabric_cost) AS max_fabric_cost,
        STRING_AGG(DISTINCT article_name, ',' ORDER BY article_name) AS article_names,
    FROM article_sku AS a_sku
    GROUP BY a_sku.sku, a_sku.sku_state, a_sku.product_id
),
article_freq AS (
    SELECT product_id, article_names, COUNT(*) AS freq
    FROM sku_fabric_costs GROUP BY product_id, article_names
),
top_article AS (
    SELECT product_id,
        ARRAY_AGG(article_names ORDER BY freq DESC LIMIT 1)[OFFSET(0)] AS most_common_article_names
    FROM article_freq GROUP BY product_id
),
avg_fabric_cost_prod AS (
    SELECT fc.product_id,
        AVG(fc.max_fabric_cost) AS max_fabric_cost,
        AVG(fc.min_fabric_cost) AS min_fabric_cost,
        t.most_common_article_names AS article_names
    FROM sku_fabric_costs AS fc
    LEFT JOIN top_article AS t ON fc.product_id = t.product_id
    GROUP BY fc.product_id, t.most_common_article_names
),
costs AS (
    SELECT
        amp.product_id, p.product_name,
        amp.apparel_manufacturer_id, amp.is_finished_product,
        amp.manufacturer_cost as manufacture_cost,
        fc.min_fabric_cost, fc.max_fabric_cost, fc.article_names,
        CASE WHEN amp.is_finished_product = True THEN amp.manufacturer_cost
             ELSE amp.manufacturer_cost + fc.max_fabric_cost END AS manufacturing_cost,
    FROM `insider-data-lake.integrated.muninn_apparel_manufacturers_products` AS amp
    LEFT JOIN avg_fabric_cost_prod AS fc ON fc.product_id = amp.product_id
    LEFT JOIN `insider-data-lake.integrated.muninn_products` AS p ON p.product_id = amp.product_id
    WHERE amp.status IN ('available','approved','incubation')
),
base_intermediaria AS (
    SELECT
        ampup.apparel_manufacturer_production_unit_id,
        am.supplier_id, amp.apparel_manufacturer_id,
        s.alias, s.city, s.state, s.created_at AS date_supplier_creation,
        p.product_id, p.product_name,
        amp.is_finished_product, amp.order_minimum_volume, amp.lead_time,
        ampu.apparel_manufacturer_cell_number,
        MAX(ampup.weekly_maximum_productive_capacity) OVER (
            PARTITION BY ampup.apparel_manufacturer_production_unit_id
        ) AS max_capacity,
        ampup.weekly_maximum_productive_capacity,
        fp.full_price, amp.status AS status_cell,
        4*ampup.weekly_maximum_productive_capacity AS monthly_capacity,
        4*MAX(ampup.weekly_maximum_productive_capacity) OVER (
            PARTITION BY ampup.apparel_manufacturer_production_unit_id
        ) AS cell_max_monthly_capacity,
        COUNT(DISTINCT am.supplier_id) OVER (PARTITION BY p.product_id) AS num_suppliers_per_product,
    FROM `insider-data-lake.integrated.muninn_apparel_manufacturer_production_units_products` AS ampup
    LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturers_products` AS amp
        ON amp.id = ampup.apparel_manufacturer_product_id
    LEFT JOIN `insider-data-lake.integrated.muninn_products` AS p ON p.product_id = amp.product_id
    LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturers` AS am ON am.id = amp.apparel_manufacturer_id
    LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturer_production_units` AS ampu
        ON ampu.id = ampup.apparel_manufacturer_production_unit_id
    LEFT JOIN `insider-data-lake.integrated.muninn_suppliers` AS s ON s.id = am.supplier_id
    LEFT JOIN full_price AS fp ON p.product_name = fp.product_name
    WHERE amp.status IN ('available','approved','incubation')
        AND ampup.weekly_maximum_productive_capacity > 0
),
cell_products AS (
    SELECT apparel_manufacturer_production_unit_id,
        COUNT(DISTINCT b.product_name) AS n_products_in_cell,
        STRING_AGG(DISTINCT b.product_name, ', ') AS products_in_cell
    FROM base_intermediaria AS b
    GROUP BY apparel_manufacturer_production_unit_id
),
sku_data AS (
    SELECT ps.sku, ps.product_sku_id, ps.sku_name, sku_d.sku_state, sku_d.product_name,
        sku_d.family, sku_d.category, psf.fabric_id, a.name AS article_name
    FROM `insider-data-lake.integrated.muninn_product_skus` AS ps
    LEFT JOIN `insider-data-lake.integrated.muninn_product_skus_fabrics` AS psf ON ps.product_sku_id = psf.product_sku_id
    LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS f ON psf.fabric_id = f.id
    LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS a ON f.article_id = a.id
    LEFT JOIN `insider-data-lake.integrated.skus` AS sku_d ON sku_d.sku = ps.sku
    ORDER BY ps.product_sku_id, psf.fabric_id
),
dpi AS (
    SELECT product_name, SUM(treated_generated_revenue) AS treated_generated_revenue
    FROM `insider-data-lake.sop_silver.demand_prediction_input`
    WHERE DATE(reference_date) >= DATE_SUB(DATE_TRUNC(CURRENT_DATE(), MONTH), INTERVAL 3 MONTH)
        AND DATE(reference_date) <  DATE_TRUNC(CURRENT_DATE(), MONTH)
        AND product_name IS NOT NULL
    GROUP BY product_name
),
revenue_totals AS (
    SELECT product_name, treated_generated_revenue,
        SUM(treated_generated_revenue) OVER () AS total_treated_generated_revenue
    FROM dpi
),
cum AS (
    SELECT product_name, treated_generated_revenue, total_treated_generated_revenue,
        SUM(treated_generated_revenue) OVER (
            ORDER BY treated_generated_revenue DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cum_treated_generated_revenue
    FROM revenue_totals
),
abc_curve AS (
    SELECT product_name, treated_generated_revenue,
        SAFE_DIVIDE(cum_treated_generated_revenue, total_treated_generated_revenue) AS cum_share,
        CASE
            WHEN SAFE_DIVIDE(cum_treated_generated_revenue, total_treated_generated_revenue) <= 0.8 THEN 'A'
            WHEN SAFE_DIVIDE(cum_treated_generated_revenue, total_treated_generated_revenue) <= 0.95 THEN 'B'
            ELSE 'C'
        END AS tag_abc
    FROM cum
),
freqs AS (
    SELECT product_name, sku_state, article_name, family, category, COUNT(*) AS freq
    FROM sku_data GROUP BY product_name, sku_state, article_name, family, category
),
product_data AS (
    SELECT f.product_name, p.product_id,
        CASE WHEN SUM(CASE WHEN f.sku_state = 'ativo_perene' THEN 1 ELSE 0 END) > 0
             THEN 'ativo_perene'
             ELSE ARRAY_AGG(f.sku_state ORDER BY freq DESC LIMIT 1)[OFFSET(0)] END AS product_state,
        ARRAY_TO_STRING(ARRAY_AGG(DISTINCT f.article_name), ', ') AS article_name,
        ARRAY_AGG(f.family ORDER BY freq DESC LIMIT 1)[OFFSET(0)] AS family,
        ARRAY_AGG(f.category ORDER BY freq DESC LIMIT 1)[OFFSET(0)] AS category,
        abc.tag_abc
    FROM freqs AS f
    LEFT JOIN `insider-data-lake.integrated.muninn_products` AS p ON f.product_name = p.product_name
    LEFT JOIN abc_curve AS abc ON abc.product_name = f.product_name
    GROUP BY f.product_name, p.product_id, abc.tag_abc
)
SELECT
    b.*,
    c.manufacturing_cost, c.article_names,
    SAFE_DIVIDE(b.full_price, c.manufacturing_cost) AS mark_up,
    MIN(c.manufacturing_cost) OVER (PARTITION BY b.product_id, b.is_finished_product) AS min_manufacturing_cost,
    cp.n_products_in_cell, cp.products_in_cell,
    pd.tag_abc, pd.product_state,
FROM base_intermediaria AS b
LEFT JOIN cell_products AS cp ON b.apparel_manufacturer_production_unit_id = cp.apparel_manufacturer_production_unit_id
LEFT JOIN costs AS c ON b.apparel_manufacturer_id = c.apparel_manufacturer_id
    AND b.product_id = c.product_id AND b.is_finished_product = c.is_finished_product
LEFT JOIN product_data AS pd ON b.product_id = pd.product_id
WHERE pd.product_state NOT IN ('desativado')
"""

print("⏳ Carregando base de capacidade...")
df_capacity = client.query(SQL_CAPACITY).to_dataframe()
print(f"✅ df_capacity: {len(df_capacity):,} linhas | {df_capacity.shape[1]} colunas")
df_capacity.head(3)

⏳ Carregando base de capacidade...


/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ df_capacity: 229 linhas | 28 colunas


,apparel_manufacturer_production_unit_id,supplier_id,apparel_manufacturer_id,alias,city,state,date_supplier_creation,product_id,product_name,is_finished_product,...,cell_max_monthly_capacity,num_suppliers_per_product,manufacturing_cost,article_names,mark_up,min_manufacturing_cost,n_products_in_cell,products_in_cell,tag_abc,product_state
0,414,17,42,FABIO,São Paulo,SP,2025-02-06 13:31:01.894,265,Performance T-shirt 2.0 Masculino,False,...,2000,2,66.493700000,Outlast,3.895105852,64.803700000,1,Performance T-shirt 2.0 Masculino,B,ativo_perene
1,102,19,54,FITMAX,Cascável,PR,2025-02-06 13:31:01.894,265,Performance T-shirt 2.0 Masculino,False,...,4000,2,64.803700000,Outlast,3.996685374,64.803700000,2,"Sportee Masculino, Performance T-shirt 2.0 Mas...",B,ativo_perene
2,62,17,42,FABIO,São Paulo,SP,2025-02-06 13:31:01.894,126,Tech T-shirt Long Sleeve Feminino,True,...,7500,4,47.350000000,Modal,5.047518479,47.350000000,2,"Tech T-shirt Long Sleeve Masculino, Tech T-shi...",B,ativo_perene


In [63]:
# Célula 2 — OPs com etapas de produção (lead time realizado + stamps por fase)
#
# Fonte principal: `insider-lake-sensitive.integrated_br.supply_production_orders`
# (uma linha por SKU/OP; agregada para uma linha por production_order_code).
# Enriquecimentos:
#   - `insider-lake-sensitive.unfolded_br.unfolded_qualita_auditoria` → data de auditoria.
#   - `insider-lake-sensitive.integrated_br.supply_warehouse_inbound_sftp` → última NF
#     importada no CD por remetente (via supplier_map alias→razão social).
#   - `insider-lake-sensitive.landing_br.muninn_production_orders_raw` → snapshots
#     diários para extrair o PRIMEIRO dia em que cada estágio aparece (granularidade
#     diária; substitui a antiga CDC `muninn_append_public.ProductionOrders`).
# A legacy `insider-data-lake.sop_silver.supply_chain_efficiency_model_input` é mantida
# como FALLBACK (COALESCE) para `dt_largest_entry_warehouse` e demais campos planejados
# até a cobertura da nova fonte estar validada em produção.
# `final_compat` projeta o contrato de colunas usado downstream (id, op_code, stamps,
# datas planejadas) para preservar compatibilidade com o restante do notebook.
SQL_OPS = """
WITH supplier_map AS (
  SELECT * FROM UNNEST([
    STRUCT('BAE BRASIL' AS apparel_manufacturer_alias, 'BB ARTIGOS DE VESTUARIO LTDA' AS razao_social_remetente),
    STRUCT('ARTIGO X', 'ARTIGOX INDUSTRIA E COMERCIO LTDA'),
    STRUCT('NOVA FORMULA', 'NOVA FORMULA INDUSTRIALIZACAO TEXTIL LTDA'),
    STRUCT('INTERTEXTIL', 'INTER TEXTIL LTDA'),
    STRUCT('DDAL', 'DDAL CONFECCOES LTDA'),
    STRUCT('ABBA', 'ABBA CONFECCOES LTDA'),
    STRUCT('FABIO', 'AMG CONFECÇÕES  LTDA'),
    STRUCT('ART LIVRE', 'ART LIVRE MODAS LTDA'),
    STRUCT('DALOP', 'DALOP CONFECCAO E COMERCIO DE ARTIGOS DO VESTUARIO LTDA'),
    STRUCT("MALHAS D'STEFANO", 'MALHAS D ESTEFANO LTDA'),
    STRUCT('PIXIE', 'PIXIE ARTEMODA EIRELI'),
    STRUCT('AZZURRA', 'AZZURRA CONFECCOES LTDA'),
    STRUCT('BY COTTON', 'BY COTTON INDUSTRIA DO VESTUARIO LTDA'),
    STRUCT('CLARA BELLA', 'CLARA BELLA CONFECCOES LTDA'),
    STRUCT('CLESTE BRASIL CONFECÇÕES LTDA', 'CLESTE BRASIL CONFECCOES LTDA'),
    STRUCT('FITMAX', 'FITMAX LTDA'),
    STRUCT('GOAT', 'GOAT CONFECCOES LTDA'),
    STRUCT('LUTESTIL', 'LUTESTIL INDUSTRIA COMERCIO DE ROUPAS LTDA'),
    STRUCT('RDM', 'MARIA ENGRACIA FLORENTINO DA SILVA LTDA'),
    STRUCT('MAURA', 'MAURA RODRIGUES DA SILVA CONFECCOES'),
    STRUCT('MC & MC', 'MC&MC CAMISARIA EIRELI'),
    STRUCT('RIZLLEP', 'RIZLLEP INDUSTRIA DE CONFECCOES LTDA'),
    STRUCT('BLUTEXTIL', 'BLUTEXTIL'),
    STRUCT('Conceitun', 'CONCEITUN IND. DE ROUPAS LTDA'),
    STRUCT('T Christina', 'CONFECCOES TCHRISTINA LTDA'),
    STRUCT('INDÚSTRIA TEXTIL BETILHA LTDA', 'INDUSTRIA TEXTIL BETILHA LTDA'),
    STRUCT('KABRIOLLI', 'KABRIOLLI IND E COM DE ROUPAS LTDA'),
    STRUCT('Lorsa', 'LORSA MODAS E CONFECCOES LTDA'),
    STRUCT('LUNELLI', 'LUNELLI COMERCIO DO VESTUARIO LTDA'),
    STRUCT('MAC CLEM', 'MAC CLEM INDUSTRIA E COMERCIO DE CONFECCOES LTDA'),
    STRUCT('MASH', 'MASH INDUSTRIA E COMERCIO LTDA'),
    STRUCT('TRICCOT', 'N TRICCOT CONFECCOES EIRELI'),
    STRUCT('NATURAL COMPANY CONFECCOES LTDA', 'NATURAL COMPANY CONFECCOES LTDA'),
    STRUCT('WARUSKY', 'WARUSKY COM. IND. E REP. LTDA')
  ])
),

base_ops_detail AS (
  SELECT
    spo.production_order_id,
    spo.production_order_code,
    REGEXP_REPLACE(
      UPPER(TRIM(CAST(spo.production_order_code AS STRING))),
      r'[^0-9A-Z]',
      ''
    ) AS op_norm,

    COALESCE(spo.unified_production_order_status, spo.production_order_status) AS current_production_stage,
    COALESCE(spo.is_finished_product_order, FALSE) AS is_finished_product_order,
    spo.product_name,
    spo.product_color,
    spo.sku_state,
    UPPER(TRIM(spo.apparel_manufacturer_alias)) AS supplier_name,
    spo.production_cycle_name AS cycle_name,
    spo.production_order_type,
    spo.production_order_status,
    spo.data_availability,
    spo.acceptance_status,

    SAFE_CAST(spo.sku_requested_quantity AS NUMERIC) AS sku_requested_quantity,
    SAFE_CAST(spo.hike_sku_finished_quantity AS NUMERIC) AS hike_sku_finished_quantity,

    DATE(spo.planned_production_start_date) AS dt_planned_production_start,
    DATE(spo.planned_production_end_date) AS dt_planned_production_end,
    DATE(spo.planned_production_delivery_date) AS dt_planned_entry_warehouse,
    DATE(spo.expected_production_delivery_date) AS dt_reviewed_entry_warehouse,
    DATE(spo.real_production_delivery_date) AS real_production_delivery_date,

    DATE(spo.production_order_created_at) AS production_order_created_at,
    DATE(spo.production_order_updated_at) AS production_order_updated_at,

    DATE(spo.fabric_reservation_date) AS data_reserva_mp,
    DATE(spo.real_fabric_receiving_date) AS data_recebimento_real_tecido,
    DATE(spo.real_production_start_date) AS data_inicio_real_producao,
    DATE(spo.planned_production_delivery_date) AS data_entrega_planejada

  FROM `insider-lake-sensitive.integrated_br.supply_production_orders` spo
  WHERE spo.planned_production_delivery_date >= '2026-01-01'
    AND spo.data_availability IN ('HIKE AND MUNINN', 'MUNINN ONLY')
    AND LOWER(spo.production_order_status) NOT IN ('pedido cancelado', 'canceled', 'encerrado')
    AND spo.production_order_code != 'OPF84N142'
),

legacy_ops_by_op AS (
  SELECT
    scemi.op_code,
    ANY_VALUE(scemi.current_production_stage) AS legacy_current_production_stage,
    ANY_VALUE(COALESCE(scemi.is_finished_product_order, FALSE)) AS legacy_is_finished_product_order,
    STRING_AGG(DISTINCT scemi.product_name, ', ') AS legacy_product_names,
    STRING_AGG(DISTINCT scemi.product_color, ', ') AS legacy_product_colors,
    STRING_AGG(DISTINCT scemi.status_sku, ', ') AS legacy_sku_status,
    ANY_VALUE(scemi.supplier_name) AS legacy_supplier_name,
    ANY_VALUE(scemi.cycle_name) AS legacy_cycle_name,
    ANY_VALUE(scemi.production_order_type) AS legacy_production_order_type,
    MAX(scemi.planned_quantity_op) AS legacy_planned_quantity_op,
    MAX(scemi.received_quantity_op) AS legacy_received_quantity_op,
    MAX(DATE(scemi.dt_planned_production_start)) AS legacy_dt_planned_production_start,
    MAX(DATE(scemi.dt_planned_production_end)) AS legacy_dt_planned_production_end,
    MAX(DATE(scemi.dt_planned_entry_warehouse)) AS legacy_dt_planned_entry_warehouse,
    MAX(DATE(scemi.dt_reviewed_entry_warehouse)) AS legacy_dt_reviewed_entry_warehouse,
    MAX(DATE(scemi.dt_largest_entry_warehouse)) AS legacy_dt_largest_entry_warehouse
  FROM `insider-data-lake.sop_silver.supply_chain_efficiency_model_input` scemi
  GROUP BY scemi.op_code
),

base_ops AS (
  SELECT
    bod.production_order_code,
    ANY_VALUE(bod.production_order_id) AS id,
    ANY_VALUE(bod.op_norm) AS op_norm,

    COALESCE(
      ANY_VALUE(bod.current_production_stage),
      ANY_VALUE(legacy.legacy_current_production_stage)
    ) AS current_production_stage,

    COALESCE(
      ANY_VALUE(bod.is_finished_product_order),
      ANY_VALUE(legacy.legacy_is_finished_product_order),
      FALSE
    ) AS is_finished_product_order,

    COALESCE(
      STRING_AGG(DISTINCT bod.product_name, ', '),
      ANY_VALUE(legacy.legacy_product_names)
    ) AS product_names,

    COALESCE(
      STRING_AGG(DISTINCT bod.product_color, ', '),
      ANY_VALUE(legacy.legacy_product_colors)
    ) AS product_colors,

    COALESCE(
      STRING_AGG(DISTINCT bod.sku_state, ', '),
      ANY_VALUE(legacy.legacy_sku_status)
    ) AS sku_status,

    COALESCE(
      ANY_VALUE(bod.supplier_name),
      UPPER(TRIM(ANY_VALUE(legacy.legacy_supplier_name)))
    ) AS supplier_name,

    COALESCE(
      ANY_VALUE(bod.cycle_name),
      ANY_VALUE(legacy.legacy_cycle_name)
    ) AS cycle_name,

    COALESCE(
      ANY_VALUE(bod.production_order_type),
      ANY_VALUE(legacy.legacy_production_order_type)
    ) AS production_order_type,

    COALESCE(
      SUM(bod.sku_requested_quantity),
      ANY_VALUE(legacy.legacy_planned_quantity_op)
    ) AS planned_quantity_op,

    COALESCE(
      SUM(bod.hike_sku_finished_quantity),
      ANY_VALUE(legacy.legacy_received_quantity_op)
    ) AS received_quantity_op,

    COALESCE(
      MAX(bod.dt_planned_production_start),
      ANY_VALUE(legacy.legacy_dt_planned_production_start)
    ) AS dt_planned_production_start,

    COALESCE(
      MAX(bod.dt_planned_production_end),
      ANY_VALUE(legacy.legacy_dt_planned_production_end)
    ) AS dt_planned_production_end,

    COALESCE(
      MAX(bod.dt_planned_entry_warehouse),
      ANY_VALUE(legacy.legacy_dt_planned_entry_warehouse)
    ) AS dt_planned_entry_warehouse,

    COALESCE(
      MAX(bod.dt_reviewed_entry_warehouse),
      ANY_VALUE(legacy.legacy_dt_reviewed_entry_warehouse)
    ) AS dt_reviewed_entry_warehouse,

    COALESCE(
      MAX(bod.real_production_delivery_date),
      ANY_VALUE(legacy.legacy_dt_largest_entry_warehouse)
    ) AS dt_largest_entry_warehouse,

    ANY_VALUE(bod.production_order_status) AS production_order_status,
    ANY_VALUE(bod.data_availability) AS data_availability,

    MIN(bod.production_order_created_at) AS production_order_created_at,
    MIN(CASE
      WHEN bod.acceptance_status IN ('accepted', 'accepted_with_changes')
      THEN bod.production_order_updated_at
    END) AS accepted_production_date,

    MAX(bod.data_entrega_planejada) AS data_entrega_planejada,
    MAX(bod.data_reserva_mp) AS data_reserva_mp,
    MAX(bod.data_recebimento_real_tecido) AS data_recebimento_real_tecido,
    MAX(bod.data_inicio_real_producao) AS data_inicio_real_producao

  FROM base_ops_detail bod
  LEFT JOIN legacy_ops_by_op legacy
    ON legacy.op_code = bod.production_order_code
  GROUP BY bod.production_order_code
),

auditoria AS (
  SELECT
    REGEXP_REPLACE(
      UPPER(TRIM(CAST(numero_original AS STRING))),
      r'[^0-9A-Z]',
      ''
    ) AS op_norm,
    MIN(SAFE.PARSE_DATE('%Y-%m-%d', LEFT(data_agendamento, 10))) AS data_auditoria
  FROM `insider-lake-sensitive.unfolded_br.unfolded_qualita_auditoria`
  GROUP BY 1
),

inbound_cd AS (
  SELECT
    UPPER(TRIM(razao_social_remetente)) AS remetente_norm,
    MAX(DATE(nf_importada_em)) AS data_ultima_nf_importada_cd
  FROM `insider-lake-sensitive.integrated_br.supply_warehouse_inbound_sftp`
  GROUP BY 1
),

muninn_first_status AS (
  SELECT
    mpo.order_code,
    mpo.status,
    DATE(mpo.ingestion_date) AS data_entrada_status
  FROM `insider-lake-sensitive.landing_br.muninn_production_orders_raw` AS mpo
  WHERE mpo.order_code LIKE 'OPF%'
    AND mpo.status IN (
      'finished',
      'canceled',
      'items_delivery_and_invoicing',
      'cut_fabric_and_sewing_process',
      'fabric_validation_and_pre_cutting',
      'waiting_fabric_arrival',
      'pending',
      'quality_inspection',
      'order_request_validation'
    )
    AND DATE(mpo.ingestion_date) >= '2026-01-01'

  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY mpo.order_code, mpo.status
    ORDER BY mpo.ingestion_date ASC
  ) = 1
),

muninn_pivot AS (
  SELECT
    order_code,

    MIN(IF(status = 'pending', data_entrada_status, NULL)) AS dt_muninn_pending,
    MIN(IF(status = 'order_request_validation', data_entrada_status, NULL)) AS dt_muninn_order_request_validation,
    MIN(IF(status = 'waiting_fabric_arrival', data_entrada_status, NULL)) AS dt_muninn_waiting_fabric_arrival,
    MIN(IF(status = 'fabric_validation_and_pre_cutting', data_entrada_status, NULL)) AS dt_muninn_fabric_validation_and_pre_cutting,
    MIN(IF(status = 'cut_fabric_and_sewing_process', data_entrada_status, NULL)) AS dt_muninn_cut_fabric_and_sewing_process,
    MIN(IF(status = 'quality_inspection', data_entrada_status, NULL)) AS dt_muninn_quality_inspection,
    MIN(IF(status = 'items_delivery_and_invoicing', data_entrada_status, NULL)) AS dt_muninn_items_delivery_and_invoicing,
    MIN(IF(status = 'finished', data_entrada_status, NULL)) AS dt_muninn_finished,
    MIN(IF(status = 'canceled', data_entrada_status, NULL)) AS dt_muninn_canceled

  FROM muninn_first_status
  GROUP BY 1
),

supplier_status AS (
  SELECT DISTINCT
    UPPER(TRIM(supplier_name)) AS supplier_name,
    supplier_relationship_status
  FROM `insider-data-lake.sop_gold.production_order_integrated`
  WHERE supplier_name IS NOT NULL
),

final_enriched AS (
  SELECT
    b.*,
    a.data_auditoria,
    ic.data_ultima_nf_importada_cd,

    m.dt_muninn_pending,
    m.dt_muninn_order_request_validation,
    m.dt_muninn_waiting_fabric_arrival,
    m.dt_muninn_fabric_validation_and_pre_cutting,
    m.dt_muninn_cut_fabric_and_sewing_process,
    m.dt_muninn_quality_inspection,
    m.dt_muninn_items_delivery_and_invoicing,
    m.dt_muninn_finished,
    m.dt_muninn_canceled,

    CASE
      WHEN DATE_DIFF(b.data_recebimento_real_tecido, b.data_reserva_mp, DAY) BETWEEN 0 AND 365
      THEN DATE_DIFF(b.data_recebimento_real_tecido, b.data_reserva_mp, DAY)
      ELSE NULL
    END AS dias_reserva_ate_recebimento_real_tecido,

    DATE_DIFF(a.data_auditoria, b.data_inicio_real_producao, DAY)
      AS dias_inicio_producao_ate_auditoria,

    CASE
      WHEN DATE_DIFF(ic.data_ultima_nf_importada_cd, a.data_auditoria, DAY) BETWEEN -5 AND 90
      THEN DATE_DIFF(ic.data_ultima_nf_importada_cd, a.data_auditoria, DAY)
      ELSE NULL
    END AS dias_auditoria_ate_entrega_cd,

    DATE_DIFF(
      m.dt_muninn_waiting_fabric_arrival,
      m.dt_muninn_order_request_validation,
      DAY
    ) AS dias_order_request_validation_ate_waiting_fabric_arrival,

    DATE_DIFF(
      m.dt_muninn_fabric_validation_and_pre_cutting,
      m.dt_muninn_waiting_fabric_arrival,
      DAY
    ) AS dias_waiting_fabric_arrival_ate_fabric_validation_and_pre_cutting,

    DATE_DIFF(
      m.dt_muninn_cut_fabric_and_sewing_process,
      m.dt_muninn_fabric_validation_and_pre_cutting,
      DAY
    ) AS dias_fabric_validation_and_pre_cutting_ate_cut_fabric_and_sewing_process,

    DATE_DIFF(
      m.dt_muninn_quality_inspection,
      m.dt_muninn_cut_fabric_and_sewing_process,
      DAY
    ) AS dias_cut_fabric_and_sewing_process_ate_quality_inspection,

    DATE_DIFF(
      m.dt_muninn_items_delivery_and_invoicing,
      m.dt_muninn_quality_inspection,
      DAY
    ) AS dias_quality_inspection_ate_items_delivery_and_invoicing,

    DATE_DIFF(
      m.dt_muninn_finished,
      m.dt_muninn_items_delivery_and_invoicing,
      DAY
    ) AS dias_items_delivery_and_invoicing_ate_finished,

    DATE_DIFF(
      m.dt_muninn_finished,
      m.dt_muninn_order_request_validation,
      DAY
    ) AS dias_total_muninn_order_request_validation_ate_finished

  FROM base_ops b
  LEFT JOIN auditoria a
    ON b.op_norm = a.op_norm
  LEFT JOIN supplier_map sm
    ON b.supplier_name = UPPER(TRIM(sm.apparel_manufacturer_alias))
  LEFT JOIN inbound_cd ic
    ON ic.remetente_norm = UPPER(TRIM(sm.razao_social_remetente))
  LEFT JOIN muninn_pivot m
    ON b.production_order_code = m.order_code
  LEFT JOIN supplier_status ss
    ON b.supplier_name = ss.supplier_name
  WHERE (
    ss.supplier_relationship_status IS NULL
    OR ss.supplier_relationship_status NOT IN ('terminated', 'discontinued')
  )
),

final_compat AS (
  SELECT
    id,
    production_order_code AS op_code,
    current_production_stage,
    is_finished_product_order,
    product_names,
    product_colors,
    sku_status,
    supplier_name,
    cycle_name,
    production_order_type,
    planned_quantity_op,
    received_quantity_op,
    dt_planned_production_start,
    dt_planned_production_end,
    dt_planned_entry_warehouse,
    dt_reviewed_entry_warehouse,
    dt_largest_entry_warehouse,

    TIMESTAMP(production_order_created_at) AS stamp_created_production_order,
    TIMESTAMP(accepted_production_date) AS stamp_accepted_production,
    TIMESTAMP(dt_muninn_order_request_validation) AS stamp_stage_order_request_validation,
    TIMESTAMP(dt_muninn_waiting_fabric_arrival) AS stamp_stage_waiting_fabric_arrival,
    TIMESTAMP(dt_muninn_fabric_validation_and_pre_cutting) AS stamp_stage_fabric_validation_and_pre_cutting,
    TIMESTAMP(dt_muninn_cut_fabric_and_sewing_process) AS stamp_stage_cut_fabric_and_sewing_process,
    TIMESTAMP(dt_muninn_quality_inspection) AS stamp_stage_quality_inspection,
    TIMESTAMP(dt_muninn_items_delivery_and_invoicing) AS stamp_stage_items_delivery_and_invoicing,
    TIMESTAMP(dt_muninn_finished) AS stamp_stage_finished,

    production_order_status,
    data_availability,
    data_entrega_planejada,
    data_reserva_mp,
    data_recebimento_real_tecido,
    data_inicio_real_producao,
    data_auditoria,
    data_ultima_nf_importada_cd,

    dt_muninn_pending,
    dt_muninn_order_request_validation,
    dt_muninn_waiting_fabric_arrival,
    dt_muninn_fabric_validation_and_pre_cutting,
    dt_muninn_cut_fabric_and_sewing_process,
    dt_muninn_quality_inspection,
    dt_muninn_items_delivery_and_invoicing,
    dt_muninn_finished,
    dt_muninn_canceled,

    dias_reserva_ate_recebimento_real_tecido,
    dias_inicio_producao_ate_auditoria,
    dias_auditoria_ate_entrega_cd,
    dias_order_request_validation_ate_waiting_fabric_arrival,
    dias_waiting_fabric_arrival_ate_fabric_validation_and_pre_cutting,
    dias_fabric_validation_and_pre_cutting_ate_cut_fabric_and_sewing_process,
    dias_cut_fabric_and_sewing_process_ate_quality_inspection,
    dias_quality_inspection_ate_items_delivery_and_invoicing,
    dias_items_delivery_and_invoicing_ate_finished,
    dias_total_muninn_order_request_validation_ate_finished
  FROM final_enriched
)

SELECT *
FROM final_compat
ORDER BY
  supplier_name,
  dt_planned_entry_warehouse,
  op_code
"""

print("⏳ Carregando OPs com timestamps de etapas...")
df_ops_raw = client.query(SQL_OPS).to_dataframe()
print(f"✅ df_ops_raw: {len(df_ops_raw):,} linhas | {df_ops_raw.shape[1]} colunas")
df_ops_raw.head(3)


⏳ Carregando OPs com timestamps de etapas...


/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ df_ops_raw: 2,633 linhas | 53 colunas


,id,op_code,current_production_stage,is_finished_product_order,product_names,product_colors,sku_status,supplier_name,cycle_name,production_order_type,...,dias_reserva_ate_recebimento_real_tecido,dias_inicio_producao_ate_auditoria,dias_auditoria_ate_entrega_cd,dias_order_request_validation_ate_waiting_fabric_arrival,dias_waiting_fabric_arrival_ate_fabric_validation_and_pre_cutting,dias_fabric_validation_and_pre_cutting_ate_cut_fabric_and_sewing_process,dias_cut_fabric_and_sewing_process_ate_quality_inspection,dias_quality_inspection_ate_items_delivery_and_invoicing,dias_items_delivery_and_invoicing_ate_finished,dias_total_muninn_order_request_validation_ate_finished
0,61817,OPF118N16,finished,True,Core T-shirt Masculino,Rain Forest,"ativo_perene, desativado",ABBA,C2-Core-extra,committed,...,21,38,68,<NA>,30,31,8,3,17,<NA>
1,61818,OPF118N17,finished,True,Core T-shirt Masculino,Vulcano,ativo_perene,ABBA,C2-Core-extra,committed,...,21,13,60,<NA>,30,40,8,14,5,<NA>
2,61819,OPF118N18,finished,True,Core T-shirt Masculino,Navy,"ativo_perene, desativado",ABBA,C2-Core-extra,committed,...,21,33,73,<NA>,30,31,3,6,2,<NA>


In [64]:
# Célula 3 — Pré-processamento: filtros, lead time realizado, macroetapas e mês de fechamento

import numpy as np

# --- 3.1 Filtros conforme metodologia do relatório original ---
df_ops = df_ops_raw.copy()

# Validação de contrato após migração da SQL_OPS
required_cols = [
    "id",
    "op_code",
    "current_production_stage",
    "is_finished_product_order",
    "product_names",
    "product_colors",
    "sku_status",
    "supplier_name",
    "cycle_name",
    "production_order_type",
    "planned_quantity_op",
    "received_quantity_op",
    "dt_planned_production_start",
    "dt_planned_production_end",
    "dt_planned_entry_warehouse",
    "dt_reviewed_entry_warehouse",
    "dt_largest_entry_warehouse",
    "stamp_created_production_order",
    "stamp_accepted_production",
    "stamp_stage_order_request_validation",
    "stamp_stage_waiting_fabric_arrival",
    "stamp_stage_fabric_validation_and_pre_cutting",
    "stamp_stage_cut_fabric_and_sewing_process",
    "stamp_stage_quality_inspection",
    "stamp_stage_items_delivery_and_invoicing",
    "stamp_stage_finished",
    "dias_order_request_validation_ate_waiting_fabric_arrival",
    "dias_waiting_fabric_arrival_ate_fabric_validation_and_pre_cutting",
    "dias_fabric_validation_and_pre_cutting_ate_cut_fabric_and_sewing_process",
    "dias_cut_fabric_and_sewing_process_ate_quality_inspection",
    "dias_quality_inspection_ate_items_delivery_and_invoicing",
    "dias_items_delivery_and_invoicing_ate_finished",
]

missing_cols = [col for col in required_cols if col not in df_ops.columns]
if missing_cols:
    raise ValueError(f"Colunas obrigatórias ausentes em df_ops_raw: {missing_cols}")

duplicated_ops = df_ops["op_code"].duplicated().sum()
if duplicated_ops > 0:
    raise ValueError(f"A nova SQL_OPS retornou OPs duplicadas: {duplicated_ops} linhas duplicadas em op_code")

print("✅ Schema obrigatório validado")
print(f"✅ Unicidade por OP validada: {df_ops['op_code'].nunique():,} OPs únicas")

# Diagnóstico de cobertura bruta (antes dos filtros downstream)
coverage_cols = [
    "dt_largest_entry_warehouse",
    "stamp_created_production_order",
    "stamp_stage_order_request_validation",
    "stamp_stage_waiting_fabric_arrival",
    "stamp_stage_fabric_validation_and_pre_cutting",
    "stamp_stage_cut_fabric_and_sewing_process",
    "stamp_stage_quality_inspection",
    "stamp_stage_items_delivery_and_invoicing",
    "stamp_stage_finished",
]
print("\nCobertura bruta antes dos filtros:")
for col in coverage_cols:
    print(f"  {col}: {df_ops[col].notna().mean():.1%}")


# Apenas production_order_type = 'committed'
df_ops = df_ops[df_ops["production_order_type"] == "committed"]

# Remover ciclos B2B e EPA
df_ops = df_ops[~df_ops["cycle_name"].str.contains("B2B|EPA", na=False, case=False)]

# Datas auxiliares usadas para fechamento e diagnóstico
df_ops["dt_largest_entry_warehouse"] = pd.to_datetime(df_ops["dt_largest_entry_warehouse"], utc=True)
df_ops["stamp_created_production_order"] = pd.to_datetime(df_ops["stamp_created_production_order"], utc=True)

# --- 3.2 Lead time realizado e macroetapas — soma real das 6 transições Muninn ---
stamp_cols = [
    "stamp_stage_order_request_validation",
    "stamp_stage_waiting_fabric_arrival",
    "stamp_stage_fabric_validation_and_pre_cutting",
    "stamp_stage_cut_fabric_and_sewing_process",
    "stamp_stage_quality_inspection",
    "stamp_stage_items_delivery_and_invoicing",
    "stamp_stage_finished",
]
for col in stamp_cols:
    df_ops[col] = pd.to_datetime(df_ops[col], utc=True)


TRANSICOES_LT_COLS = [
    "dias_order_request_validation_ate_waiting_fabric_arrival",
    "dias_waiting_fabric_arrival_ate_fabric_validation_and_pre_cutting",
    "dias_fabric_validation_and_pre_cutting_ate_cut_fabric_and_sewing_process",
    "dias_cut_fabric_and_sewing_process_ate_quality_inspection",
    "dias_quality_inspection_ate_items_delivery_and_invoicing",
    "dias_items_delivery_and_invoicing_ate_finished",
]
for col in TRANSICOES_LT_COLS:
    df_ops[col] = pd.to_numeric(df_ops[col], errors="coerce")


MACRO_ETAPAS = {
    "etapa_reserva_recebimento_tecido": {
        "label": "Reserva → Recebimento Tecido",
        "cols": [
            "dias_order_request_validation_ate_waiting_fabric_arrival",
            "dias_waiting_fabric_arrival_ate_fabric_validation_and_pre_cutting",
        ],
    },
    "etapa_producao_auditoria": {
        "label": "Produção → Auditoria",
        "cols": [
            "dias_fabric_validation_and_pre_cutting_ate_cut_fabric_and_sewing_process",
            "dias_cut_fabric_and_sewing_process_ate_quality_inspection",
        ],
    },
    "etapa_auditoria_entrega_cd": {
        "label": "Auditoria → Entrega CD",
        "cols": [
            "dias_quality_inspection_ate_items_delivery_and_invoicing",
            "dias_items_delivery_and_invoicing_ate_finished",
        ],
    },
}
ETAPAS_COLS = list(MACRO_ETAPAS.keys())
ETAPAS_LABELS = [MACRO_ETAPAS[col]["label"] for col in ETAPAS_COLS]
ETAPAS_SLA_BY_COL = {
    col: SLA_ETAPAS[MACRO_ETAPAS[col]["label"]]
    for col in ETAPAS_COLS
}


def _diff_days(end_col, start_col):
    """Dias entre dois stamps; não exige etapas intermediárias preenchidas."""
    return (
        (df_ops[end_col] - df_ops[start_col]).dt.total_seconds() / 86400.0
    ).clip(lower=0)


def _sum_duration_cols(cols):
    valores = df_ops[cols].apply(pd.to_numeric, errors="coerce").clip(lower=0)
    return valores.sum(axis=1, min_count=1)


df_ops["etapa_reserva_recebimento_tecido"] = _diff_days(
    "stamp_stage_fabric_validation_and_pre_cutting",
    "stamp_stage_order_request_validation",
)
df_ops["etapa_producao_auditoria"] = _diff_days(
    "stamp_stage_quality_inspection",
    "stamp_stage_fabric_validation_and_pre_cutting",
)
df_ops["etapa_auditoria_entrega_cd"] = _diff_days(
    "stamp_stage_finished",
    "stamp_stage_quality_inspection",
)

df_ops["lead_time_realizado"] = _diff_days(
    "stamp_stage_finished",
    "stamp_stage_order_request_validation",
)
df_ops["lead_time_realizado"] = df_ops["lead_time_realizado"].fillna(
    _sum_duration_cols(TRANSICOES_LT_COLS)
)

# Incluir somente OPs que chegaram ao fim do fluxo Muninn
df_ops = df_ops[df_ops["stamp_stage_finished"].notna()]

print("\nSanidade do lead time realizado (soma das 6 transições Muninn):")
print(f"  OPs com lead time válido: {len(df_ops):,}")
print(f"  Target total padrão: {TARGET_LEAD_TIME:.0f} dias")
print(f"  Lead time mínimo: {df_ops['lead_time_realizado'].min():.0f} dias")
print(f"  Lead time mediano: {df_ops['lead_time_realizado'].median():.1f} dias")
print(f"  Lead time p75: {df_ops['lead_time_realizado'].quantile(0.75):.1f} dias")
print(f"  Lead time máximo: {df_ops['lead_time_realizado'].max():.0f} dias")

if df_ops["lead_time_realizado"].median() <= 0:
    raise ValueError("Lead time mediano inválido: esperado valor positivo")


print(f"OPs após filtros: {len(df_ops):,}")
print(f"  Triangulação: {(~df_ops['is_finished_product_order']).sum():,}")
print(f"  Produto acabado: {df_ops['is_finished_product_order'].sum():,}")


print("\nCobertura por macroetapa após cálculo:")
for col, label in zip(ETAPAS_COLS, ETAPAS_LABELS):
    cobertura = df_ops[col].notna().mean()
    mediana = df_ops[col].median()
    sla = ETAPAS_SLA_BY_COL[col]
    print(f"  {label} ({col}): cobertura={cobertura:.1%} | mediana={mediana:.1f} dias | SLA={sla}d")


# --- 3.3 Mês de fechamento (para séries temporais e janelas móveis) ---
df_ops["mes_fechamento"] = (
    df_ops["stamp_stage_finished"].dt.tz_convert(None).dt.to_period("M").dt.to_timestamp()
)

# --- 3.4 Enriquecer com lead time teórico da base de capacidade ---
capacity_lt = (
    df_capacity[["alias", "product_name", "is_finished_product", "lead_time", "tag_abc"]]
    .drop_duplicates()
    .rename(columns={
        "alias": "supplier_name",
        "lead_time": "lead_time_teorico",
        "is_finished_product": "is_finished_product_order",
    })
)
capacity_lt["is_finished_product_order"] = capacity_lt["is_finished_product_order"].astype(bool)

df_ops = df_ops.merge(
    capacity_lt,
    how="left",
    left_on=["supplier_name", "product_names", "is_finished_product_order"],
    right_on=["supplier_name", "product_name", "is_finished_product_order"],
)

df_ops["desvio_lt"] = df_ops["lead_time_realizado"] - df_ops["lead_time_teorico"]
df_ops["dentro_do_prazo"] = df_ops["lead_time_realizado"] <= TARGET_LEAD_TIME

# --- 3.5 Buckets de volume (consome CONFIG quando definido; fallback para defaults) ---
_bins   = CONFIG["buckets_volume"] if "CONFIG" in globals() else [0, 200, 500, 1000, 2000, 5000, float("inf")]
_labels = CONFIG["labels_volume"]  if "CONFIG" in globals() else ["<200", "200-499", "500-999", "1000-1999", "2000-4999", "5000+"]
df_ops["volume_bucket"] = pd.cut(df_ops["planned_quantity_op"], bins=_bins, labels=_labels, right=False)

# --- 3.6 Separar fluxos (re-derivado APÓS todas as colunas calculadas) ---
df_tri = df_ops[~df_ops["is_finished_product_order"]].copy()
df_pa  = df_ops[df_ops["is_finished_product_order"]].copy()

print(f"\n✅ Pré-processamento completo.")
print(f"   df_tri (triangulação): {len(df_tri):,} OPs")
print(f"   df_pa  (produto acabado): {len(df_pa):,} OPs")
coverage_lt_teorico = df_ops["lead_time_teorico"].notna().mean()
print(f"   Cobertura lead time teórico: {coverage_lt_teorico:.1%}")

if coverage_lt_teorico < 0.30:
    exemplos_sem_match = (
        df_ops[df_ops["lead_time_teorico"].isna()]
        [["supplier_name", "product_names", "is_finished_product_order"]]
        .drop_duplicates()
        .head(20)
    )
    print("\n⚠️  Atenção: baixa cobertura de lead time teórico. Exemplos sem match:")
    try:
        from IPython.display import display
        display(exemplos_sem_match)
    except Exception:
        print(exemplos_sem_match.to_string())
print(f"   Cobertura das 6 etapas (todas preenchidas): {df_ops[ETAPAS_COLS].notna().all(axis=1).mean():.1%}")
print(f"     ↳ PA:  {df_pa[ETAPAS_COLS].notna().all(axis=1).mean():.1%}")
print(f"     ↳ Tri: {df_tri[ETAPAS_COLS].notna().all(axis=1).mean():.1%}")


✅ Schema obrigatório validado
✅ Unicidade por OP validada: 2,633 OPs únicas

Cobertura bruta antes dos filtros:
  dt_largest_entry_warehouse: 42.5%
  stamp_created_production_order: 100.0%
  stamp_stage_order_request_validation: 39.6%
  stamp_stage_waiting_fabric_arrival: 72.1%
  stamp_stage_fabric_validation_and_pre_cutting: 52.2%
  stamp_stage_cut_fabric_and_sewing_process: 54.3%
  stamp_stage_quality_inspection: 40.7%
  stamp_stage_items_delivery_and_invoicing: 42.6%
  stamp_stage_finished: 40.4%

Sanidade do lead time realizado (soma das 6 transições Muninn):
  OPs com lead time válido: 631
  Target total padrão: 124 dias
  Lead time mínimo: 0 dias
  Lead time mediano: 71.5 dias
  Lead time p75: 112.0 dias
  Lead time máximo: 133 dias
OPs após filtros: 631
  Triangulação: 126
  Produto acabado: 505

Cobertura por macroetapa após cálculo:
  Reserva → Recebimento Tecido (etapa_reserva_recebimento_tecido): cobertura=5.1% | mediana=59.5 dias | SLA=65d
  Produção → Auditoria (etapa_prod

In [65]:
# Célula 4 — Nível 1: KPI Cards (com variação MoM)
from IPython.display import display, HTML

# Valor atual (último mês fechado completo)
mes_atual = df_ops["mes_fechamento"].max()
mes_anterior = (mes_atual - pd.DateOffset(months=1))


def mediana_mes(df, mes):
    sub = df[df["mes_fechamento"] == mes]
    return sub["lead_time_realizado"].median() if len(sub) >= 10 else None


lt_geral_atual = df_ops["lead_time_realizado"].median()
lt_tri_atual   = df_tri["lead_time_realizado"].median()
lt_pa_atual    = df_pa["lead_time_realizado"].median()
pct_dentro     = df_ops["dentro_do_prazo"].mean() * 100

# Variação MoM
lt_geral_ant = mediana_mes(df_ops, mes_anterior)
lt_tri_ant   = mediana_mes(df_tri, mes_anterior)
lt_pa_ant    = mediana_mes(df_pa,  mes_anterior)


def delta_mom(atual, anterior):
    if anterior is None or pd.isna(anterior) or pd.isna(atual):
        return ""
    delta = atual - anterior
    arrow = "▼" if delta < 0 else ("▲" if delta > 0 else "→")
    color = COLORS["status_ok"] if delta < 0 else (COLORS["status_critico"] if delta > 0 else COLORS["neutro_escuro"])
    return f"<span style='color:{color}'>{arrow} {abs(delta):.0f}d MoM</span>"


def kpi_card(label, value, unit="", color="#374151", sub=None):
    sub_html = f"<div style='font-size:12px;color:#666;margin-top:6px'>{sub}</div>" if sub else ""
    return f"""
    <div style='display:inline-block;background:#fafafa;border:1px solid #e5e7eb;
                border-radius:10px;padding:18px 26px;margin:8px;min-width:170px;text-align:center'>
        <div style='font-size:12px;color:#6b7280;font-weight:600;text-transform:uppercase;letter-spacing:0.5px'>{label}</div>
        <div style='font-size:34px;font-weight:700;color:{color};margin-top:4px'>{value}<span style='font-size:14px;font-weight:500'>{unit}</span></div>
        {sub_html}
    </div>"""


# Cor de status para %120d
if pct_dentro >= 65:
    cor_pct = COLORS["status_ok"]
elif pct_dentro >= 50:
    cor_pct = COLORS["status_atencao"]
else:
    cor_pct = COLORS["status_critico"]

cards_html = "".join([
    kpi_card("Mediana Geral",        f"{lt_geral_atual:.0f}", "d", COLORS["neutro_escuro"],
             f"Target: {TARGET_LEAD_TIME}d &nbsp;|&nbsp; {delta_mom(lt_geral_atual, lt_geral_ant)}"),
    kpi_card("Mediana Triangulação", f"{lt_tri_atual:.0f}",   "d", COLORS["fluxo_tri"],
             delta_mom(lt_tri_atual, lt_tri_ant)),
    kpi_card("Mediana Produto Acabado", f"{lt_pa_atual:.0f}", "d", COLORS["fluxo_pa"],
             delta_mom(lt_pa_atual, lt_pa_ant)),
    kpi_card("Dentro do Prazo (120d)", f"{pct_dentro:.1f}", "%", cor_pct,
             f"Janela: últimos {CONFIG['janela_meses']}m"),
])
display(HTML(f"""
<div style='font-family:-apple-system,sans-serif'>
    <h3 style='color:#374151;margin-bottom:8px'>📊 Nível 1 — Visão Executiva</h3>
    {cards_html}
</div>
"""))


In [66]:
# Célula 5 — Nível 1: Série Temporal Mensal por Fluxo
# Substitui o snapshot Mediana×P75 estático: o último ponto da série já entrega o snapshot
# e ainda dá tendência. Faixa P25–P75 sombreada + barras de %120d no eixo secundário.

# Cortar pela janela configurada
data_corte = pd.Timestamp.now(tz="UTC").normalize() - pd.DateOffset(months=CONFIG["janela_meses"])
df_ts = df_ops[df_ops["mes_fechamento"] >= data_corte.tz_localize(None)].copy()


def serie_por_fluxo(df_subset):
    agg = (
        df_subset.groupby("mes_fechamento")
        .agg(
            mediana=("lead_time_realizado", "median"),
            p25=("lead_time_realizado", lambda x: x.quantile(0.25)),
            p75=("lead_time_realizado", lambda x: x.quantile(0.75)),
            pct_no_prazo=("dentro_do_prazo", lambda x: x.mean() * 100),
            n_ops=("op_code", "count"),
        )
        .reset_index()
    )
    # Filtro de robustez: meses com < 3 OPs viram NaN
    agg.loc[agg["n_ops"] < 3, ["mediana", "p25", "p75", "pct_no_prazo"]] = None
    return agg


ts_tri = serie_por_fluxo(df_ts[~df_ts["is_finished_product_order"]])
ts_pa  = serie_por_fluxo(df_ts[df_ts["is_finished_product_order"]])

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Triangulação", "Produto Acabado"),
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    horizontal_spacing=0.12,
)


def add_fluxo(fig, ts, col, cor_fluxo, nome):
    # Faixa P25–P75 (sombreada)
    fig.add_trace(go.Scatter(
        x=list(ts["mes_fechamento"]) + list(ts["mes_fechamento"])[::-1],
        y=list(ts["p75"]) + list(ts["p25"])[::-1],
        fill="toself", fillcolor=cor_fluxo, opacity=0.15,
        line=dict(width=0), showlegend=False, hoverinfo="skip",
        name=f"P25–P75 {nome}",
    ), row=1, col=col, secondary_y=False)

    # Linha mediana (forte)
    fig.add_trace(go.Scatter(
        x=ts["mes_fechamento"], y=ts["mediana"], mode="lines+markers",
        line=dict(color=cor_fluxo, width=3), marker=dict(size=8),
        name=f"Mediana {nome}", legendgroup=nome,
    ), row=1, col=col, secondary_y=False)

    # Linha P75 (pontilhada)
    fig.add_trace(go.Scatter(
        x=ts["mes_fechamento"], y=ts["p75"], mode="lines",
        line=dict(color=cor_fluxo, width=1.5, dash="dot"),
        name=f"P75 {nome}", legendgroup=nome,
    ), row=1, col=col, secondary_y=False)

    # Barras %dentro120d no eixo secundário
    fig.add_trace(go.Bar(
        x=ts["mes_fechamento"], y=ts["pct_no_prazo"],
        marker_color=cor_fluxo, opacity=0.25,
        name=f"% no prazo {nome}", legendgroup=nome,
    ), row=1, col=col, secondary_y=True)

    # Linha target 120d
    fig.add_hline(y=TARGET_LEAD_TIME, line_dash="dash",
                  line_color=COLORS["target_line"], row=1, col=col, secondary_y=False)

    # Anotação da variação MoM no último ponto válido
    validos = ts.dropna(subset=["mediana"])
    if len(validos) >= 2:
        ultimo = validos.iloc[-1]
        penultimo = validos.iloc[-2]
        delta = ultimo["mediana"] - penultimo["mediana"]
        arrow = "▼" if delta < 0 else "▲"
        cor_anot = COLORS["status_ok"] if delta < 0 else COLORS["status_critico"]
        fig.add_annotation(
            x=ultimo["mes_fechamento"], y=ultimo["mediana"],
            text=f"{arrow} {abs(delta):.0f}d MoM",
            showarrow=True, arrowhead=2, ax=30, ay=-30,
            font=dict(color=cor_anot, size=11, family="sans-serif"),
            row=1, col=col,
        )


add_fluxo(fig, ts_tri, col=1, cor_fluxo=COLORS["fluxo_tri"], nome="Tri")
add_fluxo(fig, ts_pa,  col=2, cor_fluxo=COLORS["fluxo_pa"],  nome="PA")

fig.update_xaxes(title_text="Mês", row=1, col=1)
fig.update_xaxes(title_text="Mês", row=1, col=2)
fig.update_yaxes(title_text="Dias", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="Dias", row=1, col=2, secondary_y=False)
fig.update_yaxes(title_text="% no prazo", row=1, col=1, secondary_y=True, range=[0, 100])
fig.update_yaxes(title_text="% no prazo", row=1, col=2, secondary_y=True, range=[0, 100])

fig.update_layout(
    title=f"Evolução do Lead Time — últimos {CONFIG['janela_meses']} meses",
    template=TEMPLATE, height=480,
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
)
fig.show()


In [67]:
# Célula 6 — Nível 2: Decomposição de Lead Time por Etapa (6 etapas, PA e Tri)
# Parâmetro: trocar para "PA", "Tri" ou "Ambos"
FLUXO_FILTRO = "Ambos"

# Escala sequencial para as 6 etapas (gradient azul claro → escuro)
ETAPAS_CORES = ["#DBEAFE", "#93C5FD", "#60A5FA", "#3B82F6", "#1D4ED8", "#1E3A8A"]


def montar_etapas(df_subset, label_fluxo):
    if len(df_subset) == 0:
        return None
    agg = (
        df_subset.groupby("supplier_name")
        .agg(
            **{col: (col, "median") for col in ETAPAS_COLS},
            n_ops=("op_code", "count"),
            lt_total=("lead_time_realizado", "median"),
        )
        .reset_index()
    )
    agg = agg[agg["n_ops"] >= CONFIG["min_ops_grafico"]]
    agg = agg.nlargest(CONFIG["top_n_fornecedores_grafico"], "n_ops")
    agg["fluxo"] = label_fluxo
    # Ordenar por lt_total ascendente → maior LDT no topo do gráfico (barh)
    agg = agg.sort_values("lt_total", ascending=True)
    return agg


dfs_para_plotar = []
if FLUXO_FILTRO in ("PA", "Ambos"):
    pa_etapas = montar_etapas(df_pa, "PA")
    if pa_etapas is not None:
        dfs_para_plotar.append(pa_etapas)
if FLUXO_FILTRO in ("Tri", "Ambos"):
    tri_etapas = montar_etapas(df_tri, "Tri")
    if tri_etapas is not None:
        dfs_para_plotar.append(tri_etapas)

if not dfs_para_plotar:
    print(f"⚠ Sem dados suficientes para o filtro FLUXO_FILTRO={FLUXO_FILTRO!r}")
else:
    n_subplots = len(dfs_para_plotar)
    fig = make_subplots(
        rows=1, cols=n_subplots,
        subplot_titles=[f"{d['fluxo'].iloc[0]} (top {len(d)} por volume)" for d in dfs_para_plotar],
        shared_yaxes=False,
        horizontal_spacing=0.18,
    )

    for idx, agg in enumerate(dfs_para_plotar, start=1):
        for i, (col, label) in enumerate(zip(ETAPAS_COLS, ETAPAS_LABELS)):
            fig.add_trace(go.Bar(
                y=agg["supplier_name"],
                x=agg[col],
                name=label,
                orientation="h",
                marker_color=ETAPAS_CORES[i],
                text=[f"{v:.0f}d" if v >= 10 else "" for v in agg[col]],
                textposition="inside",
                insidetextanchor="middle",
                textfont=dict(size=10, color="white" if i >= 3 else "#374151"),
                hovertemplate=f"<b>%{{y}}</b><br>{label}: %{{x:.0f}}d<extra></extra>",
                showlegend=(idx == 1),
                legendgroup=label,
            ), row=1, col=idx)

        fig.add_vline(x=TARGET_LEAD_TIME, line_dash="dash",
                      line_color=COLORS["target_line"], row=1, col=idx)
        fig.update_xaxes(title_text="Dias (mediana)", row=1, col=idx)

    fig.update_layout(
        title=f"Decomposição de Lead Time por Etapa — {FLUXO_FILTRO}",
        barmode="stack", template=TEMPLATE,
        height=max(450, 30 * max(len(d) for d in dfs_para_plotar) + 100),
        legend=dict(orientation="h", yanchor="bottom", y=-0.18, xanchor="center", x=0.5),
    )
    fig.show()


In [68]:
# Célula 6.1 — Tabela detalhada de etapas: Fornecedor × Produto × Fluxo
# Reproduz o formato da tabela "Gargalo por etapa" da guilda de LDT.

PRODUTO_FILTRO = None         # None = todos | ou string parcial (ex: "Tech T-shirt")
FLUXO_FILTRO_TABELA = "Ambos" # "PA", "Tri" ou "Ambos"
MIN_OPS_TABELA = 3

_agg_spec = {col: (col, "median") for col in ETAPAS_COLS}
_agg_spec["n_ops"] = ("op_code", "count")
_agg_spec["lt_total"] = ("lead_time_realizado", "median")

base = df_ops.copy()
if FLUXO_FILTRO_TABELA == "PA":
    base = base[base["is_finished_product_order"] == True]
elif FLUXO_FILTRO_TABELA == "Tri":
    base = base[base["is_finished_product_order"] == False]
if PRODUTO_FILTRO:
    base = base[base["product_names"].str.contains(PRODUTO_FILTRO, case=False, na=False)]

tabela = (
    base.groupby(["supplier_name", "product_names", "is_finished_product_order"], dropna=False)
    .agg(**_agg_spec)
    .reset_index()
)
tabela = tabela[tabela["n_ops"] >= MIN_OPS_TABELA].copy()
tabela["fluxo"] = tabela["is_finished_product_order"].map({True: "PA", False: "Tri"})

# Identificar gargalo principal por linha
tabela["gargalo_etapa"] = tabela[ETAPAS_COLS].idxmax(axis=1).map(dict(zip(ETAPAS_COLS, ETAPAS_LABELS)))
tabela["gargalo_dias"] = tabela[ETAPAS_COLS].max(axis=1)
tabela["gargalo_principal"] = (
    tabela["gargalo_etapa"] + " (" + tabela["gargalo_dias"].round(0).astype("Int64").astype(str) + "d)"
)

# Ordenar por lt_total decrescente
tabela = tabela.sort_values("lt_total", ascending=False)

# Truncar nome de produto
tabela["produto"] = tabela["product_names"].astype(str).str[:35]

# Renomear colunas de etapas para os labels finais
rename_map = dict(zip(ETAPAS_COLS, ETAPAS_LABELS))
tabela = tabela.rename(columns=rename_map)

display_cols = ["supplier_name", "produto", "fluxo", "n_ops", "lt_total",
                *ETAPAS_LABELS, "gargalo_principal"]


def color_fluxo(v):
    if v == "PA":
        return f"background-color:{COLORS['fluxo_pa']}; color:white; font-weight:600; text-align:center"
    if v == "Tri":
        return f"background-color:{COLORS['fluxo_tri']}; color:#374151; font-weight:600; text-align:center"
    return ""


print(f"Tabela: {len(tabela)} pares (fornecedor × produto × fluxo) — filtro: {FLUXO_FILTRO_TABELA}, mín {MIN_OPS_TABELA} OPs")
(tabela[display_cols]
    .style
    .background_gradient(subset=ETAPAS_LABELS, cmap="Blues", axis=None)
    .map(color_fluxo, subset=["fluxo"])
    .format({"lt_total": "{:.0f}d", **{l: "{:.0f}d" for l in ETAPAS_LABELS}})
    .hide(axis="index")
)


Tabela: 58 pares (fornecedor × produto × fluxo) — filtro: Ambos, mín 3 OPs


/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_22427/606006249.py:29: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  tabela["gargalo_etapa"] = tabela[ETAPAS_COLS].idxmax(axis=1).map(dict(zip(ETAPAS_COLS, ETAPAS_LABELS)))


supplier_name,produto,fluxo,n_ops,lt_total,Reserva → Recebimento Tecido,Produção → Auditoria,Auditoria → Entrega CD,gargalo_principal
ART LIVRE,Tech T-shirt Gola U Feminino,PA,6,133d,87d,32d,44d,Reserva → Recebimento Tecido (87d)
ART LIVRE,Tech T-shirt Heavy Slim Masculino,PA,4,133d,nand,48d,15d,Produção → Auditoria (48d)
ART LIVRE,Tech T-shirt Long Sleeve Feminino,PA,7,133d,nand,51d,49d,Produção → Auditoria (51d)
GOAT,Wingsuit Feminino,Tri,12,129d,60d,34d,22d,Reserva → Recebimento Tecido (60d)
RIZLLEP,Saia Envelope Breeze Feminino,PA,8,129d,nand,62d,64d,Auditoria → Entrega CD (64d)
RDM,Calcinha Brief Feminino,PA,4,129d,nand,35d,31d,Produção → Auditoria (35d)
RIZLLEP,Daily T-shirt Masculino,PA,25,127d,67d,40d,38d,Reserva → Recebimento Tecido (67d)
LUTESTIL,Calcinha Minimal Corte a Laser Femi,PA,8,118d,0d,120d,7d,Produção → Auditoria (120d)
MAURA,Calça FutureForm Masculino,Tri,20,116d,nand,37d,36d,Produção → Auditoria (37d)
DALOP,Regata Intech Masculino,PA,10,113d,58d,50d,24d,Reserva → Recebimento Tecido (58d)


In [69]:
# Célula 7 — Nível 2: Aderência ao Lead Time Teórico por Fornecedor
# Barras coloridas por status (verde/laranja/vermelho) + tabela companion com %120d.

aderencia = (
    df_ops[df_ops["lead_time_teorico"].notna()]
    .groupby(["supplier_name", "is_finished_product_order"])
    .agg(
        desvio_mediano=("desvio_lt", "median"),
        n_ops=("op_code", "count"),
        lt_realizado=("lead_time_realizado", "median"),
        lt_teorico=("lead_time_teorico", "median"),
        pct_no_prazo=("dentro_do_prazo", lambda x: x.mean() * 100),
    )
    .reset_index()
)
aderencia["fluxo"] = aderencia["is_finished_product_order"].map({True: "PA", False: "Tri"})
aderencia = aderencia[aderencia["n_ops"] >= 5].sort_values("desvio_mediano", ascending=False)


def cor_status_desvio(d):
    if d > CONFIG["threshold_desvio_critico"]:
        return COLORS["status_critico"]
    if d > CONFIG["threshold_desvio_atencao"]:
        return COLORS["status_atencao"]
    if d < 0:
        return COLORS["neutro_claro"]
    return COLORS["status_ok"]


aderencia["cor"] = aderencia["desvio_mediano"].apply(cor_status_desvio)
aderencia["label"] = aderencia["supplier_name"] + " (" + aderencia["fluxo"] + ")"

fig = go.Figure(go.Bar(
    x=aderencia["label"],
    y=aderencia["desvio_mediano"],
    marker_color=aderencia["cor"],
    text=aderencia["desvio_mediano"].round(0).astype("Int64").astype(str) + "d",
    textposition="outside",
    customdata=aderencia[["lt_realizado", "lt_teorico", "n_ops", "pct_no_prazo"]].values,
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Desvio: %{y:.0f}d<br>"
        "Realizado: %{customdata[0]:.0f}d | Teórico: %{customdata[1]:.0f}d<br>"
        "% dentro 120d: %{customdata[3]:.1f}%<br>"
        "n OPs: %{customdata[2]}<extra></extra>"
    ),
))
fig.add_hline(y=CONFIG["threshold_desvio_critico"], line_dash="dash",
              line_color=COLORS["status_critico"], annotation_text="Crítico >30d")
fig.add_hline(y=CONFIG["threshold_desvio_atencao"], line_dash="dot",
              line_color=COLORS["status_atencao"], annotation_text="Atenção >10d")
fig.add_hline(y=0, line_color="black", line_width=0.5)
fig.update_layout(
    title="Desvio Mediano: Realizado vs. Teórico por Fornecedor × Fluxo",
    template=TEMPLATE, yaxis_title="Desvio (dias)", xaxis_tickangle=-40,
    height=480,
)
fig.show()

# --- Tabela companion ---
def status_label(d):
    if d > CONFIG["threshold_desvio_critico"]:
        return "🔴 Crítico"
    if d > CONFIG["threshold_desvio_atencao"]:
        return "🟠 Atenção"
    if d < 0:
        return "⚪ Abaixo do teórico"
    return "🟢 OK"


aderencia_tab = aderencia.copy()
aderencia_tab["status"] = aderencia_tab["desvio_mediano"].apply(status_label)
aderencia_tab["desvio_pct"] = (
    (aderencia_tab["desvio_mediano"] / aderencia_tab["lt_teorico"] * 100)
    .round(0).astype("Int64").astype(str) + "%"
)

display_cols = ["supplier_name", "fluxo", "n_ops", "lt_realizado", "lt_teorico",
                "desvio_mediano", "desvio_pct", "pct_no_prazo", "status"]


def color_fluxo_cell(v):
    if v == "PA":
        return f"background-color:{COLORS['fluxo_pa']}; color:white; font-weight:600; text-align:center"
    if v == "Tri":
        return f"background-color:{COLORS['fluxo_tri']}; color:#374151; font-weight:600; text-align:center"
    return ""


(aderencia_tab[display_cols]
    .style
    .map(color_fluxo_cell, subset=["fluxo"])
    .format({
        "lt_realizado": "{:.0f}d", "lt_teorico": "{:.0f}d",
        "desvio_mediano": "{:+.0f}d", "pct_no_prazo": "{:.1f}%",
    })
    .background_gradient(subset=["desvio_mediano"], cmap="RdYlGn_r", vmin=-30, vmax=90)
    .hide(axis="index")
)


supplier_name,fluxo,n_ops,lt_realizado,lt_teorico,desvio_mediano,desvio_pct,pct_no_prazo,status
GOAT,Tri,14,129d,40d,+89d,222%,35.7%,🔴 Crítico
MAURA,Tri,32,107d,50d,+58d,115%,59.4%,🔴 Crítico
FABIO,Tri,15,92d,35d,+57d,163%,100.0%,🔴 Crítico
ART LIVRE,Tri,12,93d,56d,+51d,91%,66.7%,🔴 Crítico
RIZLLEP,PA,25,127d,84d,+43d,51%,48.0%,🔴 Crítico
MALHAS D'STEFANO,PA,57,99d,60d,+39d,65%,91.2%,🔴 Crítico
PIXIE,Tri,18,98d,60d,+38d,63%,83.3%,🔴 Crítico
ART LIVRE,PA,68,133d,100d,+33d,33%,45.6%,🔴 Crítico
DALOP,PA,12,113d,90d,+23d,26%,58.3%,🟠 Atenção
ABBA,PA,17,93d,90d,+3d,3%,88.2%,🟢 OK


In [70]:
# Célula 8.1 — Nível 1: LDT mediano por bucket de volume × fluxo (agregado executivo)
matrix = (
    df_ops.groupby(["volume_bucket", "is_finished_product_order"], observed=True)["lead_time_realizado"]
    .agg(["median", "count"])
    .reset_index()
)
matrix.columns = ["volume_bucket", "is_finished_product_order", "mediana", "n_ops"]
matrix["fluxo"] = matrix["is_finished_product_order"].map({True: "Produto Acabado", False: "Triangulação"})
matrix = matrix[matrix["n_ops"] >= CONFIG["min_ops_grafico"]]

fig = px.bar(
    matrix, x="volume_bucket", y="mediana", color="fluxo", barmode="group",
    text=matrix.apply(lambda r: f"{r['mediana']:.0f}d (n={r['n_ops']})" if r['n_ops'] >= 10 else f"{r['mediana']:.0f}d", axis=1),
    color_discrete_map={
        "Triangulação": COLORS["fluxo_tri"],
        "Produto Acabado": COLORS["fluxo_pa"],
    },
    labels={"mediana": "Lead Time Mediano (dias)", "volume_bucket": "Faixa de Volume"},
    title="LDT Mediano por Faixa de Volume × Fluxo",
    template=TEMPLATE,
    category_orders={"volume_bucket": CONFIG["labels_volume"]},
)
fig.add_hline(y=TARGET_LEAD_TIME, line_dash="dash", line_color=COLORS["target_line"],
              annotation_text=f"Target {TARGET_LEAD_TIME}d")
fig.update_traces(textposition="outside")
fig.update_layout(height=420)
fig.show()


In [71]:
# Célula 8.2 — Nível 2: Heatmap Fornecedor × Faixa de Volume (PA e Tri lado a lado)


def montar_heatmap(df_subset, label):
    top_forn = (
        df_subset.groupby("supplier_name")["op_code"].count()
        .nlargest(CONFIG["top_n_fornecedores_heatmap"]).index.tolist()
    )
    pivot_mediana = (
        df_subset[df_subset["supplier_name"].isin(top_forn)]
        .groupby(["supplier_name", "volume_bucket"], observed=True)["lead_time_realizado"]
        .median().unstack("volume_bucket")
        .reindex(columns=CONFIG["labels_volume"])
    )
    pivot_n = (
        df_subset[df_subset["supplier_name"].isin(top_forn)]
        .groupby(["supplier_name", "volume_bucket"], observed=True)["op_code"]
        .count().unstack("volume_bucket")
        .reindex(columns=CONFIG["labels_volume"])
    )
    # Mascarar células com n_ops < min_ops_grafico
    pivot_mediana = pivot_mediana.where(pivot_n >= CONFIG["min_ops_grafico"])
    # Ordenar por mediana geral do fornecedor (asc = melhor no topo)
    ordem = pivot_mediana.median(axis=1).sort_values(ascending=True).index
    return pivot_mediana.loc[ordem], pivot_n.loc[ordem], label


hm_tri, n_tri, _ = montar_heatmap(df_tri, "Triangulação")
hm_pa,  n_pa,  _ = montar_heatmap(df_pa,  "Produto Acabado")

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Triangulação", "Produto Acabado"),
                    horizontal_spacing=0.18)

for idx, (hm, n_df, _) in enumerate([(hm_tri, n_tri, "Tri"), (hm_pa, n_pa, "PA")], start=1):
    texto = hm.copy().astype(object)
    for i in hm.index:
        for j in hm.columns:
            v = hm.loc[i, j]
            n_val = n_df.loc[i, j] if not pd.isna(n_df.loc[i, j]) else 0
            texto.loc[i, j] = f"{v:.0f}d<br>n={int(n_val)}" if pd.notna(v) else "—"

    fig.add_trace(go.Heatmap(
        z=hm.values, x=list(hm.columns), y=list(hm.index),
        text=texto.values, texttemplate="%{text}",
        textfont=dict(size=10),
        colorscale=[
            [0.0,  COLORS["status_ok"]],
            [0.4,  "#FEF3C7"],
            [0.5,  COLORS["target_line"]],
            [0.6,  COLORS["status_atencao"]],
            [1.0,  COLORS["status_critico"]],
        ],
        zmid=TARGET_LEAD_TIME,
        zmin=30, zmax=210,
        showscale=(idx == 2),
        colorbar=dict(title="LDT (d)", x=1.02) if idx == 2 else None,
        hovertemplate="<b>%{y}</b><br>Volume: %{x}<br>LDT: %{z:.0f}d<extra></extra>",
    ), row=1, col=idx)

fig.update_layout(
    title="Heatmap: Lead Time Mediano por Fornecedor × Faixa de Volume",
    template=TEMPLATE,
    height=max(500, 25 * max(len(hm_tri), len(hm_pa)) + 100),
)
fig.update_xaxes(title_text="Faixa de Volume")
fig.show()


In [72]:
# Célula 8.5 — Nível 3: Tabela de Recomendação de Prazo por Fornecedor × Produto × Fluxo
# Output direto da ação P1 da guilda: substituir o prazo padrão de 120d pelo prazo real recomendado.
# Granularidade: fornecedor × product_names × fluxo (PA / Tri)

PERCENTIL        = CONFIG["percentil_recomendacao"]
JANELA_LONGA     = CONFIG["janela_meses"]
JANELA_CURTA     = CONFIG["janela_meses_curta"]
MIN_OPS_REC      = 3  # mínimo local — mais permissivo que o global de 5,
                       # pois granularidade por produto reduz n por célula


def recomendacao_por_produto(df_subset, janela_meses, min_ops=MIN_OPS_REC):
    data_corte = (
        pd.Timestamp.now(tz="UTC").normalize() - pd.DateOffset(months=janela_meses)
    ).tz_localize(None)
    sub = df_subset[df_subset["mes_fechamento"] >= data_corte].copy()

    agg = (
        sub.groupby(["supplier_name", "product_names", "is_finished_product_order"])
        .agg(
            n_ops                  = ("op_code",             "count"),
            lt_realizado_p50       = ("lead_time_realizado", "median"),
            lt_recomendado_raw     = ("lead_time_realizado", lambda x: x.quantile(PERCENTIL)),
            pct_no_prazo           = ("dentro_do_prazo",     lambda x: x.mean() * 100),
            lt_teorico_cadastrado  = ("lead_time_teorico",   "median"),
        )
        .reset_index()
    )
    agg = agg[agg["n_ops"] >= min_ops].copy()
    agg["lt_recomendado"] = (agg["lt_recomendado_raw"] / 5).round() * 5
    return agg.drop(columns="lt_recomendado_raw")


rec_longa = recomendacao_por_produto(df_ops, JANELA_LONGA)
rec_curta = recomendacao_por_produto(df_ops, JANELA_CURTA)[
    ["supplier_name", "product_names", "is_finished_product_order",
     "lt_recomendado", "n_ops"]
].rename(columns={
    "lt_recomendado": "lt_recomendado_curta",
    "n_ops":          "n_ops_curta",
})

rec = rec_longa.merge(
    rec_curta,
    on=["supplier_name", "product_names", "is_finished_product_order"],
    how="left",
)
rec["fluxo"] = rec["is_finished_product_order"].map({True: "PA", False: "Tri"})

rec["delta_vs_cadastrado"] = (
    rec["lt_recomendado"] - rec["lt_teorico_cadastrado"]
).round(0)


def direcao(d):
    if pd.isna(d):   return "—"
    if d > 10:       return "▲ Aumentar"
    if d < -10:      return "▼ Reduzir"
    return           "→ Manter"


def confianca(n):
    if n >= 30: return "🟢 Alta"
    if n >= 10: return "🟡 Média"
    return      "🔴 Baixa"


def tendencia(row):
    if (pd.isna(row.get("lt_recomendado_curta"))
            or row.get("n_ops_curta", 0) < 3):
        return "—"
    diff = row["lt_recomendado_curta"] - row["lt_recomendado"]
    if diff < -5:  return "📉 Melhorando"
    if diff > 5:   return "📈 Piorando"
    return         "≡ Estável"


rec["direcao"]      = rec["delta_vs_cadastrado"].apply(direcao)
rec["confianca"]    = rec["n_ops"].apply(confianca)
rec["tendencia_3m"] = rec.apply(tendencia, axis=1)

rec["produto"] = rec["product_names"].str[:40]

rec = rec.sort_values(
    "delta_vs_cadastrado",
    key=lambda x: x.abs(),
    ascending=False,
)

display_cols = [
    "supplier_name", "produto", "fluxo", "n_ops",
    "lt_teorico_cadastrado", "lt_recomendado", "lt_recomendado_curta",
    "delta_vs_cadastrado", "direcao", "pct_no_prazo",
    "tendencia_3m", "confianca",
]


def color_fluxo_cell(v):
    if v == "PA":
        return (f"background-color:{COLORS['fluxo_pa']};"
                "color:white; font-weight:600; text-align:center")
    if v == "Tri":
        return (f"background-color:{COLORS['fluxo_tri']};"
                "color:#374151; font-weight:600; text-align:center")
    return ""


def color_direcao(v):
    if "Aumentar" in str(v):
        return f"background-color:{COLORS['status_critico']}; color:white; font-weight:600"
    if "Reduzir"  in str(v):
        return f"background-color:{COLORS['status_ok']}; color:white; font-weight:600"
    return ""


print(
    f"📋 Recomendação de Prazo — P{int(PERCENTIL*100)} | janela {JANELA_LONGA}m "
    f"(curta: {JANELA_CURTA}m) | mín {MIN_OPS_REC} OPs por par"
)
print(f"   Granularidade: fornecedor × produto × fluxo")
print(f"   Pares com amostra suficiente: {len(rec)}")
print(
    f"   ▲ Aumentar: {(rec['direcao']=='▲ Aumentar').sum()} | "
    f"▼ Reduzir: {(rec['direcao']=='▼ Reduzir').sum()} | "
    f"→ Manter: {(rec['direcao']=='→ Manter').sum()}"
)

(rec[display_cols]
    .style
    .map(color_fluxo_cell,  subset=["fluxo"])
    .map(color_direcao,     subset=["direcao"])
    .background_gradient(
        subset=["delta_vs_cadastrado"],
        cmap="RdYlGn_r",
        vmin=-40,
        vmax=90,
    )
    .format({
        "lt_teorico_cadastrado": "{:.0f}d",
        "lt_recomendado":        "{:.0f}d",
        "lt_recomendado_curta":  "{:.0f}d",
        "delta_vs_cadastrado":   "{:+.0f}d",
        "pct_no_prazo":          "{:.1f}%",
    }, na_rep="—")
    .hide(axis="index")
)


📋 Recomendação de Prazo — P75 | janela 12m (curta: 3m) | mín 3 OPs por par
   Granularidade: fornecedor × produto × fluxo
   Pares com amostra suficiente: 58
   ▲ Aumentar: 23 | ▼ Reduzir: 14 | → Manter: 9


supplier_name,produto,fluxo,n_ops,lt_teorico_cadastrado,lt_recomendado,lt_recomendado_curta,delta_vs_cadastrado,direcao,pct_no_prazo,tendencia_3m,confianca
RDM,Calcinha Minimal Feminino,PA,4,120d,10d,—,-110d,▼ Reduzir,100.0%,—,🔴 Baixa
GOAT,Wingsuit Feminino,Tri,12,40d,130d,130d,+90d,▲ Aumentar,33.3%,≡ Estável,🟡 Média
MAURA,Calça FutureForm Masculino,Tri,20,50d,130d,130d,+80d,▲ Aumentar,50.0%,≡ Estável,🟡 Média
MAURA,Saia Midi Kyoto Feminino,Tri,8,49d,120d,120d,+71d,▲ Aumentar,75.0%,≡ Estável,🔴 Baixa
FABIO,Core T-shirt Masculino,Tri,12,35d,105d,110d,+70d,▲ Aumentar,100.0%,≡ Estável,🟡 Média
PIXIE,Vestido Tube Dress Curto Feminino,PA,3,90d,20d,—,-70d,▼ Reduzir,100.0%,—,🔴 Baixa
ART LIVRE,Tech T-shirt Long Sleeve Masculino,Tri,4,56d,115d,115d,+59d,▲ Aumentar,75.0%,≡ Estável,🔴 Baixa
MASH,Cueca Seamless Touch Masculino,PA,6,90d,35d,—,-55d,▼ Reduzir,100.0%,—,🔴 Baixa
ARTIGO X,Boné Outdoors,PA,4,120d,65d,—,-55d,▼ Reduzir,100.0%,—,🔴 Baixa
AZZURRA,Future Shorts 200 Masculino,PA,4,90d,35d,—,-55d,▼ Reduzir,100.0%,—,🔴 Baixa


In [73]:
# Célula 9 — Nível 3: Risco Single-Source (matriz + tabela fornecedor × produto)

# Identificar produtos single-source
single_source_produtos = df_capacity[df_capacity["num_suppliers_per_product"] == 1][
    ["product_name", "alias", "is_finished_product", "lead_time", "tag_abc"]
].drop_duplicates().rename(columns={"alias": "fornecedor", "lead_time": "lt_teorico"})

# Enriquecer com dados de OPs (lead time realizado, desvio, % no prazo)
ops_por_par = (
    df_ops.groupby(["product_names", "supplier_name", "is_finished_product_order"])
    .agg(
        lt_realizado=("lead_time_realizado", "median"),
        desvio_mediano=("desvio_lt", "median"),
        pct_no_prazo=("dentro_do_prazo", lambda x: x.mean() * 100),
        n_ops=("op_code", "count"),
    )
    .reset_index()
    .rename(columns={
        "product_names": "product_name",
        "supplier_name": "fornecedor",
        "is_finished_product_order": "is_finished_product",
    })
)

single_source = single_source_produtos.merge(
    ops_por_par, how="left",
    on=["product_name", "fornecedor", "is_finished_product"],
)
single_source["fluxo"] = single_source["is_finished_product"].map({True: "PA", False: "Tri"})


def nivel(row):
    if row.get("tag_abc") in ["A", "B"]:
        if pd.notna(row.get("desvio_mediano")) and row["desvio_mediano"] > CONFIG["threshold_desvio_critico"]:
            return "🔴 Alto"
        if pd.notna(row.get("desvio_mediano")) and row["desvio_mediano"] > CONFIG["threshold_desvio_atencao"]:
            return "🟠 Moderado"
        return "🟡 Acompanhar"
    return "🟢 Baixo"


single_source["nivel_atencao"] = single_source.apply(nivel, axis=1)

# ====== MATRIZ DE RISCO (scatter) ======
matriz = single_source.dropna(subset=["desvio_mediano", "n_ops"]).copy()

simbolo_abc = {"A": "circle", "B": "square", "C": "diamond"}
matriz["simbolo"] = matriz["tag_abc"].map(simbolo_abc).fillna("x")

fig = go.Figure()

for abc in ["A", "B", "C"]:
    sub = matriz[matriz["tag_abc"] == abc]
    if len(sub) == 0:
        continue
    fig.add_trace(go.Scatter(
        x=sub["n_ops"], y=sub["desvio_mediano"],
        mode="markers",
        marker=dict(
            symbol=simbolo_abc[abc],
            size=sub["n_ops"].clip(5, 50),
            color=sub["pct_no_prazo"],
            colorscale=[[0.0, COLORS["status_critico"]],
                        [0.5, COLORS["status_atencao"]],
                        [1.0, COLORS["status_ok"]]],
            cmin=0, cmax=100,
            colorbar=dict(title="% dentro 120d", x=1.02) if abc == "A" else None,
            showscale=(abc == "A"),
            line=dict(width=1, color="#374151"),
        ),
        name=f"Tag {abc}",
        text=sub["fornecedor"].astype(str) + " — " + sub["product_name"].astype(str).str[:30] + " (" + sub["fluxo"].astype(str) + ")",
        hovertemplate=(
            "<b>%{text}</b><br>"
            "n_OPs: %{x}<br>"
            "Desvio: %{y:.0f}d<br>"
            f"Tag ABC: {abc}<extra></extra>"
        ),
    ))

fig.add_hline(y=CONFIG["threshold_desvio_critico"], line_dash="dash",
              line_color=COLORS["status_critico"], annotation_text="Crítico >30d")
fig.add_hline(y=CONFIG["threshold_desvio_atencao"], line_dash="dot",
              line_color=COLORS["status_atencao"], annotation_text="Atenção >10d")
fig.add_hline(y=0, line_color="#374151", line_width=0.5)

fig.update_layout(
    title="Matriz de Risco Single-Source — Par Fornecedor × Produto",
    template=TEMPLATE,
    xaxis=dict(title="Volume (n_OPs últimos 12m) — log", type="log"),
    yaxis=dict(title="Desvio mediano vs teórico (dias)"),
    height=520,
    legend=dict(title="Criticidade ABC"),
)
fig.show()

# ====== TABELA fornecedor × produto ======
display_cols = ["product_name", "fornecedor", "fluxo", "tag_abc",
                "lt_teorico", "lt_realizado", "desvio_mediano", "pct_no_prazo",
                "n_ops", "nivel_atencao"]


def color_fluxo_cell(v):
    if v == "PA":
        return f"background-color:{COLORS['fluxo_pa']}; color:white; font-weight:600; text-align:center"
    if v == "Tri":
        return f"background-color:{COLORS['fluxo_tri']}; color:#374151; font-weight:600; text-align:center"
    return ""


def color_atencao(v):
    if "Alto" in str(v):
        return "background-color:#FEE2E2; font-weight:600"
    if "Moderado" in str(v):
        return "background-color:#FFEDD5"
    if "Acompanhar" in str(v):
        return "background-color:#FEF3C7"
    return ""


ordem_atencao = {"🔴 Alto": 0, "🟠 Moderado": 1, "🟡 Acompanhar": 2, "🟢 Baixo": 3}
single_source["_ord"] = single_source["nivel_atencao"].map(ordem_atencao)
single_source = single_source.sort_values(
    ["_ord", "tag_abc", "desvio_mediano"],
    ascending=[True, True, False],
).drop(columns="_ord")

print(f"📋 Pares Single-Source (fornecedor × produto): {len(single_source)}")
(single_source[display_cols]
    .style
    .map(color_fluxo_cell, subset=["fluxo"])
    .map(color_atencao, subset=["nivel_atencao"])
    .format({
        "lt_teorico": "{:.0f}d",
        "lt_realizado": "{:.0f}d",
        "desvio_mediano": "{:+.0f}d",
        "pct_no_prazo": "{:.1f}%",
    }, na_rep="—")
    .hide(axis="index")
)


📋 Pares Single-Source (fornecedor × produto): 92


product_name,fornecedor,fluxo,tag_abc,lt_teorico,lt_realizado,desvio_mediano,pct_no_prazo,n_ops,nivel_atencao
Wingsuit Feminino,GOAT,Tri,A,40d,129d,+89d,33.3%,12.000000,🔴 Alto
Calça FutureForm Masculino,MAURA,Tri,A,50d,116d,+66d,50.0%,20.000000,🔴 Alto
Maxi Saia NYIN Feminino,PIXIE,Tri,A,60d,98d,+38d,83.3%,18.000000,🔴 Alto
Saia Midi Kyoto Feminino,MAURA,Tri,A,49d,85d,+36d,75.0%,8.000000,🔴 Alto
Techsture Vest Feminino,GOAT,Tri,B,40d,98d,+58d,50.0%,2.000000,🔴 Alto
Spectrum Socks Low 2.0,MALHAS D'STEFANO,PA,B,60d,100d,+40d,93.3%,15.000000,🔴 Alto
Cueca Boxer Comfort Anti Suor Masculino,BAE BRASIL,PA,A,75d,50d,-25d,84.6%,14.000000,🟡 Acompanhar
Future Shorts 200 Masculino,AZZURRA,PA,A,90d,33d,-57d,100.0%,4.000000,🟡 Acompanhar
Camisa FutureForm Masculino,MC & MC,PA,A,90d,—,—,—,—,🟡 Acompanhar
Vestido Chemise Sem Mangas FutureForm Feminino,PIXIE,PA,A,90d,—,—,—,—,🟡 Acompanhar


In [74]:
# Célula 10 — Nível 3: Top Spreads — Mesmo Produto + Faixa de Volume
# Parte A: Gráfico de barras horizontais (top 20 por spread bruto)
# Parte B: Tabela companion sem coluna oportunidade_ops_dias

# ====== PREPARAÇÃO DOS DADOS (compartilhada com heatmap abaixo) ======

spread_base = (
    df_ops.groupby(
        ["product_names", "volume_bucket", "is_finished_product_order", "supplier_name"],
        observed=True,
    )
    .agg(
        lt_mediano = ("lead_time_realizado", "median"),
        n_ops      = ("op_code",             "count"),
    )
    .reset_index()
)
spread_base = spread_base[spread_base["n_ops"] >= CONFIG["min_ops_grafico"]]

contagem_forn = (
    spread_base.groupby(
        ["product_names", "volume_bucket", "is_finished_product_order"], observed=True
    )["supplier_name"].nunique().reset_index(name="n_fornecedores")
)
spread_base = spread_base.merge(
    contagem_forn,
    on=["product_names", "volume_bucket", "is_finished_product_order"],
)
spread_base = spread_base[spread_base["n_fornecedores"] >= 2]

spread_min = (
    spread_base.sort_values("lt_mediano")
    .groupby(
        ["product_names", "volume_bucket", "is_finished_product_order"], observed=True
    )
    .first().reset_index()
    [["product_names", "volume_bucket", "is_finished_product_order",
      "lt_mediano", "supplier_name", "n_ops"]]
    .rename(columns={
        "lt_mediano":    "lt_rapido",
        "supplier_name": "fornecedor_rapido",
        "n_ops":         "n_rapido",
    })
)

spread_max = (
    spread_base.sort_values("lt_mediano", ascending=False)
    .groupby(
        ["product_names", "volume_bucket", "is_finished_product_order"], observed=True
    )
    .first().reset_index()
    [["product_names", "volume_bucket", "is_finished_product_order",
      "lt_mediano", "supplier_name", "n_ops"]]
    .rename(columns={
        "lt_mediano":    "lt_lento",
        "supplier_name": "fornecedor_lento",
        "n_ops":         "n_lento",
    })
)

n_total_grupo = (
    spread_base.groupby(
        ["product_names", "volume_bucket", "is_finished_product_order"], observed=True
    )["n_ops"].sum().reset_index(name="n_total_grupo")
)

spread_agg = (
    spread_min
    .merge(spread_max,    on=["product_names", "volume_bucket", "is_finished_product_order"])
    .merge(n_total_grupo, on=["product_names", "volume_bucket", "is_finished_product_order"])
)
spread_agg["spread_dias"] = spread_agg["lt_lento"] - spread_agg["lt_rapido"]
spread_agg["fluxo"]       = spread_agg["is_finished_product_order"].map({True: "PA", False: "Tri"})

# ====== PARTE A: GRÁFICO DE BARRAS (top 20 por spread bruto) ======

spread_top = spread_agg.sort_values("spread_dias", ascending=False).head(20).copy()

spread_top["label"] = (
    spread_top["product_names"].str[:28] + " | "
    + spread_top["volume_bucket"].astype(str) + " | "
    + spread_top["fluxo"]
)

fig = go.Figure(go.Bar(
    x=spread_top["spread_dias"],
    y=spread_top["label"],
    orientation="h",
    marker=dict(
        color=spread_top["spread_dias"],
        colorscale=[
            [0.0, COLORS["status_ok"]],
            [0.5, COLORS["status_atencao"]],
            [1.0, COLORS["status_critico"]],
        ],
        cmin=0, cmax=150,
        showscale=False,
    ),
    text=spread_top["spread_dias"].round(0).astype("Int64").astype(str) + "d",
    textposition="outside",
    customdata=spread_top[[
        "fornecedor_rapido", "lt_rapido",
        "fornecedor_lento",  "lt_lento",
        "n_total_grupo",
    ]].values,
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Rápido: %{customdata[0]} (%{customdata[1]:.0f}d)<br>"
        "Lento:  %{customdata[2]} (%{customdata[3]:.0f}d)<br>"
        "n total no grupo: %{customdata[4]}<br>"
        "Spread: %{x:.0f}d<extra></extra>"
    ),
))
fig.update_layout(
    title="Top 20 Spreads — Oportunidade de Realocação (mesmo produto + faixa de volume)",
    template=TEMPLATE,
    xaxis_title="Spread de Lead Time (dias)",
    height=620,
    yaxis={"categoryorder": "total ascending"},
)
fig.show()

# ====== PARTE B: TABELA COMPANION (sem oportunidade_ops_dias) ======

display_cols_tab = [
    "product_names", "fluxo", "volume_bucket",
    "fornecedor_rapido", "lt_rapido", "n_rapido",
    "fornecedor_lento",  "lt_lento",  "n_lento",
    "spread_dias", "n_total_grupo",
]


def color_fluxo_cell(v):
    if v == "PA":
        return (f"background-color:{COLORS['fluxo_pa']};"
                "color:white; font-weight:600; text-align:center")
    if v == "Tri":
        return (f"background-color:{COLORS['fluxo_tri']};"
                "color:#374151; font-weight:600; text-align:center")
    return ""


(spread_top[display_cols_tab]
    .style
    .map(color_fluxo_cell, subset=["fluxo"])
    .background_gradient(subset=["spread_dias"], cmap="RdYlGn_r", vmin=0, vmax=150)
    .format({
        "lt_rapido":   "{:.0f}d",
        "lt_lento":    "{:.0f}d",
        "spread_dias": "{:.0f}d",
    })
    .hide(axis="index")
)


product_names,fluxo,volume_bucket,fornecedor_rapido,lt_rapido,n_rapido,fornecedor_lento,lt_lento,n_lento,spread_dias,n_total_grupo
Spectrum Socks Mid 2.0,PA,1000-1999,BAE BRASIL,0d,5,MALHAS D'STEFANO,107d,4,107d,9
Spectrum Socks High 2.0,PA,1000-1999,BAE BRASIL,6d,4,MALHAS D'STEFANO,96d,6,90d,10
Core T-shirt Masculino,Tri,1000-1999,NOVA FORMULA,32d,4,FABIO,98d,11,66d,15
Daily T-shirt Masculino,PA,2000-4999,ART LIVRE,93d,17,RIZLLEP,129d,13,36d,30
Tech T-shirt Gola U Masculino,Tri,2000-4999,BAE BRASIL,34d,7,BY COTTON,67d,3,34d,10
Calcinha Minimal Feminino,PA,500-999,RDM,9d,4,BAE BRASIL,40d,4,31d,8
Daily T-shirt Masculino,PA,1000-1999,RIZLLEP,64d,12,ART LIVRE,93d,9,29d,21
Tech T-shirt Gola U Masculino,PA,1000-1999,BY COTTON,54d,10,BAE BRASIL,69d,5,14d,15
Tech T-shirt Gola U Masculino,PA,2000-4999,BAE BRASIL,46d,22,BY COTTON,60d,25,14d,47


In [75]:
# Célula 10.1 — Heatmap de Spread por Produto × Faixa de Volume (PA e Tri separados)
# Ordenação crescente: produtos com menor spread ficam no topo.
# Células sem amostra suficiente exibem "—" em cinza.


def montar_heatmap_spread(df_spread_agg, label_fluxo):
    sub = df_spread_agg[df_spread_agg["fluxo"] == label_fluxo].copy()
    if len(sub) == 0:
        return None, None

    sub["produto_label"] = sub["product_names"].str[:35]

    pivot_spread = (
        sub.pivot_table(
            index="produto_label",
            columns="volume_bucket",
            values="spread_dias",
            aggfunc="median",
        )
        .reindex(columns=CONFIG["labels_volume"])
    )
    pivot_n = (
        sub.pivot_table(
            index="produto_label",
            columns="volume_bucket",
            values="n_total_grupo",
            aggfunc="sum",
        )
        .reindex(columns=CONFIG["labels_volume"])
    )

    pivot_spread = pivot_spread.where(pivot_n >= CONFIG["min_ops_grafico"])

    ordem = pivot_spread.max(axis=1).sort_values(ascending=True).index
    return pivot_spread.loc[ordem], pivot_n.loc[ordem]


hm_tri, n_tri_hm = montar_heatmap_spread(spread_agg, "Tri")
hm_pa,  n_pa_hm  = montar_heatmap_spread(spread_agg, "PA")

subplots_data = [
    (hm, n_hm, lbl)
    for hm, n_hm, lbl in [(hm_tri, n_tri_hm, "Triangulação"), (hm_pa, n_pa_hm, "Produto Acabado")]
    if hm is not None and len(hm) > 0
]

n_cols = len(subplots_data)
fig_hm = make_subplots(
    rows=1, cols=n_cols,
    subplot_titles=[lbl for _, _, lbl in subplots_data],
    horizontal_spacing=0.18,
)

for idx, (hm, n_hm, lbl) in enumerate(subplots_data, start=1):
    texto = hm.copy().astype(object)
    for i in hm.index:
        for j in hm.columns:
            v = hm.loc[i, j]
            texto.loc[i, j] = f"{v:.0f}d" if pd.notna(v) else "—"

    fig_hm.add_trace(
        go.Heatmap(
            z=hm.values,
            x=list(hm.columns),
            y=list(hm.index),
            text=texto.values,
            texttemplate="%{text}",
            textfont=dict(size=10, color="#374151"),
            colorscale=[
                [0.0,  COLORS["status_ok"]],
                [0.33, "#FEF9C3"],
                [0.66, COLORS["status_atencao"]],
                [1.0,  COLORS["status_critico"]],
            ],
            zmid=50,
            zmin=0,
            zmax=150,
            showscale=(idx == n_cols),
            colorbar=dict(
                title="Spread (d)",
                x=1.02,
                tickvals=[0, 30, 60, 90, 120, 150],
                ticktext=["0d", "30d", "60d", "90d", "120d", "150d+"],
            ) if idx == n_cols else None,
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Volume: %{x}<br>"
                "Spread: %{z:.0f}d<extra></extra>"
            ),
        ),
        row=1, col=idx,
    )
    fig_hm.update_xaxes(title_text="Faixa de Volume", row=1, col=idx)

fig_hm.update_layout(
    title=(
        "Heatmap de Spread de Lead Time por Produto × Faixa de Volume<br>"
        "<sup>Ordenado por spread crescente — verde = menor spread</sup>"
    ),
    template=TEMPLATE,
    height=max(500, 22 * max(len(hm) for hm, _, _ in subplots_data) + 120),
)
fig_hm.show()


/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_22427/270170204.py:14: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  sub.pivot_table(
/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_22427/270170204.py:23: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  sub.pivot_table(
/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_22427/270170204.py:14: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  sub.pivot_table(
/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_22427/27017020